In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/saifeldenismail/phishing-detection-dataset/Phishing Detection Dataset/Dataset.csv
/kaggle/input/datasets/saifeldenismail/top-1m/top-1m.csv
/kaggle/input/datasets/saifeldenismail/legitphish/LegitPhish Dataset/url_features_extracted1.csv
/kaggle/input/datasets/saifeldenismail/pulldd-phishing-url-low-latency-detection-dataset/cybersiren_lowlatency_dataset(1).csv
/kaggle/input/datasets/saifeldenismail/phishing-urls-uci/Phishing Websites Features.docx
/kaggle/input/datasets/saifeldenismail/phishing-urls-uci/Training Dataset.arff
/kaggle/input/datasets/saifeldenismail/phishing-urls-uci/.old.arff
/kaggle/input/datasets/shashwatwork/phishing-dataset-for-machine-learning/Phishing_Legitimate_full.csv
/kaggle/input/datasets/vanamabilashreddy/phishing-detection-urls/PhiUSIIL_Phishing_URL_Dataset.csv


In [1]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║  CyberSiren — Low-Latency Feature Extraction Pipeline                ║
║  Extracts 30 URL-structural features from DS2 + DS3                  ║
║  Based on: Chiew (2019), Tamal (2024), Mohammad (2012),              ║
║            Potpelwar (2025), Prasad & Chandra (2024)                 ║
║  Output: cybersiren_lowlatency_dataset.csv                           ║
╚══════════════════════════════════════════════════════════════════════╝

PREREQUISITE FILES:
──────────────────────────────────────────────────────────────────
 1. top-1m.csv — Cisco Umbrella top 1M domains
    → Already mounted as Kaggle dataset: saifeldenismail/top-1m
    → Path: /kaggle/input/datasets/saifeldenismail/top-1m/top-1m.csv
    → Format: rank,domain (no header) e.g. "1,google.com"
    → Used to compute TLDLegitimateProb (PhiUSIIL §3.1.4)
    → Also used to compute URLCharProb lookup table (PhiUSIIL Eq. 1)

 If you don't have this file, the script auto-generates fallback tables
 from the legitimate URLs in DS2+DS3 (less ideal but functional).
"""

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tldextract"])

import pandas as pd
import numpy as np
import re
import math
import os
import warnings
from urllib.parse import urlparse
from collections import Counter
from typing import Optional

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════

# Kaggle paths for the two compatible datasets (have raw URLs + labels)
DS2_PATH = "/kaggle/input/datasets/vanamabilashreddy/phishing-detection-urls/PhiUSIIL_Phishing_URL_Dataset.csv"
DS3_PATH = "/kaggle/input/datasets/saifeldenismail/legitphish/LegitPhish Dataset/url_features_extracted1.csv"

# Optional: Cisco Umbrella top-1M for TLD/char probability lookup tables
TOP1M_PATH = "/kaggle/input/datasets/saifeldenismail/top-1m/top-1m.csv"

OUTPUT_PATH = "/kaggle/working/cybersiren_lowlatency_dataset.csv"

# ═══════════════════════════════════════════════════════════════
# PHASE 0 — LOAD & UNIFY TO (url, label) ONLY
# ═══════════════════════════════════════════════════════════════

print("=" * 80)
print("PHASE 0: Loading datasets and unifying to (url, label)")
print("=" * 80)

# DS2: PhiUSIIL — URL col = "URL", label col = "label" (1=legit, 0=phishing)
ds2 = pd.read_csv(DS2_PATH, low_memory=False)
print(f"  DS2 (PhiUSIIL) loaded: {ds2.shape[0]:,} rows × {ds2.shape[1]} cols")
ds2_slim = ds2[["URL", "label"]].rename(columns={"URL": "url", "label": "label"})
# PhiUSIIL: label=1 means legitimate, label=0 means phishing
# We unify to: 0=legitimate, 1=phishing
ds2_slim["label"] = ds2_slim["label"].map({1: 0, 0: 1})
ds2_slim["source"] = "PhiUSIIL"
print(f"  DS2 after strip: {ds2_slim.shape[0]:,} rows — label encoding: 0=legit, 1=phish")

# DS3: LegitPhish — URL col = "URL", label col = "ClassLabel" (0=phish, 1=legit)
ds3 = pd.read_csv(DS3_PATH, low_memory=False)
print(f"  DS3 (LegitPhish) loaded: {ds3.shape[0]:,} rows × {ds3.shape[1]} cols")
ds3_slim = ds3[["URL", "ClassLabel"]].rename(columns={"URL": "url", "ClassLabel": "label"})
# LegitPhish: ClassLabel=0 means phishing, ClassLabel=1 means legitimate
# We unify to: 0=legitimate, 1=phishing
ds3_slim["label"] = ds3_slim["label"].map({1: 0, 0: 1})
ds3_slim["source"] = "LegitPhish"
print(f"  DS3 after strip: {ds3_slim.shape[0]:,} rows — label encoding: 0=legit, 1=phish")

# Combine
combined = pd.concat([ds2_slim, ds3_slim], ignore_index=True)
print(f"\n  Combined: {combined.shape[0]:,} rows")

# ═══════════════════════════════════════════════════════════════
# PHASE 1 — DATA QUALITY & DEDUPLICATION
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 1: Data quality checks & deduplication")
print("=" * 80)

# Drop nulls in url or label
before = len(combined)
combined = combined.dropna(subset=["url", "label"])
print(f"  Dropped {before - len(combined):,} rows with null url/label")

# Drop rows where URL is clearly not a URL (too short, no dots)
before = len(combined)
combined = combined[combined["url"].str.len() >= 10]
combined = combined[combined["url"].str.contains(r"\.", na=False)]
print(f"  Dropped {before - len(combined):,} rows with malformed URLs (<10 chars or no dots)")

# Normalize URLs: strip whitespace, lowercase for dedup comparison
combined["url_clean"] = combined["url"].str.strip()
# Keep original case for feature extraction, use lowercase for dedup
combined["url_lower"] = combined["url_clean"].str.lower()

# Deduplicate on exact URL match (case-insensitive)
before = len(combined)
# If same URL appears with conflicting labels, keep the one from
# PhiUSIIL (larger, more recent dataset)
source_priority = {"PhiUSIIL": 0, "LegitPhish": 1}
combined["_priority"] = combined["source"].map(source_priority)
combined = combined.sort_values("_priority").drop_duplicates(subset=["url_lower"], keep="first")
combined = combined.drop(columns=["_priority", "url_lower"])
combined = combined.reset_index(drop=True)
print(f"  Deduplicated: removed {before - len(combined):,} duplicate URLs")
print(f"  Remaining: {len(combined):,} unique URLs")

# Class distribution
vc = combined["label"].value_counts()
print(f"\n  Class distribution:")
print(f"    Legitimate (0): {vc.get(0, 0):>10,} ({vc.get(0, 0)/len(combined)*100:.1f}%)")
print(f"    Phishing   (1): {vc.get(1, 0):>10,} ({vc.get(1, 0)/len(combined)*100:.1f}%)")
print(f"    Source distribution:")
print(combined["source"].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════
# PHASE 2 — BUILD LOOKUP TABLES
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 2: Building lookup tables for derived features")
print("=" * 80)

# ---------- 2a. Character probability table (PhiUSIIL §3.1.4, Eq. 1) ----------
# "We count the occurrence of each alphabet and digit in the 10 million
#  legitimate URLs and divide them by the total count of all alphabets
#  and digits of 10 million legitimate URLs."

print("\n  [2a] Building URLCharProb lookup table...")

if os.path.exists(TOP1M_PATH):
    print(f"       Using Cisco Umbrella top-1M from {TOP1M_PATH}")
    top1m = pd.read_csv(TOP1M_PATH, header=None, names=["rank", "domain"])
    legit_url_corpus = top1m["domain"].dropna().str.lower().str.cat(sep="")
else:
    print("       top-1m.csv not found — falling back to legitimate URLs from DS2+DS3")
    legit_urls = combined[combined["label"] == 0]["url_clean"]
    legit_url_corpus = legit_urls.str.lower().str.cat(sep="")

# Count each a-z and 0-9
char_counts = Counter(c for c in legit_url_corpus if c.isalnum())
total_alnum = sum(char_counts.values())

# Build probability dict: prob(char) = count(char) / total_alnum_chars
CHAR_PROB_TABLE = {}
for c in "abcdefghijklmnopqrstuvwxyz0123456789":
    CHAR_PROB_TABLE[c] = char_counts.get(c, 0) / total_alnum if total_alnum > 0 else 0.0

print(f"       Corpus size: {total_alnum:,} alphanumeric chars")
print(f"       Top-5 chars: {sorted(CHAR_PROB_TABLE.items(), key=lambda x: -x[1])[:5]}")

# ---------- 2b. TLD legitimacy probability (PhiUSIIL §3.1.4) ----------
# "We extracted all the TLDs from the top 10 million websites and counted
#  the occurrence of each TLD."

print("\n  [2b] Building TLDLegitimateProb lookup table...")

try:
    import tldextract
    HAS_TLDEXTRACT = True
    print("       tldextract available — using for accurate TLD parsing")
except ImportError:
    HAS_TLDEXTRACT = False
    print("       tldextract not available — using fallback TLD parser")


def is_ip(hostname: str) -> bool:
    """Must be defined before fallback parsers that call it."""
    if not hostname:
        return False
    h = hostname.strip("[]")
    return bool(re.match(r"^(?:\d{1,3}\.){3}\d{1,3}$|^0x[0-9a-fA-F]+$", h, re.IGNORECASE))


def extract_tld_fallback(url: str) -> str:
    """Fallback TLD extractor without tldextract."""
    try:
        hostname = urlparse(url if "://" in url else f"http://{url}").hostname or ""
        if is_ip(hostname):
            return ""
        parts = hostname.rstrip(".").split(".")
        if len(parts) >= 2:
            # Handle double TLDs like .co.uk, .com.au
            known_double = {"co.uk", "com.au", "co.in", "co.jp", "com.br",
                           "co.za", "com.mx", "org.uk", "net.au", "ac.uk",
                           "gov.uk", "co.kr", "com.cn", "com.tw", "co.nz"}
            if len(parts) >= 3:
                candidate = f"{parts[-2]}.{parts[-1]}"
                if candidate in known_double:
                    return candidate
            return parts[-1]
        return ""
    except Exception:
        return ""


def extract_tld(url: str) -> str:
    if HAS_TLDEXTRACT:
        ext = tldextract.extract(url)
        return ext.suffix.lower() if ext.suffix else ""
    return extract_tld_fallback(url)


def extract_domain(url: str) -> str:
    if HAS_TLDEXTRACT:
        ext = tldextract.extract(url)
        return ext.domain.lower() if ext.domain else ""
    try:
        hostname = urlparse(url if "://" in url else f"http://{url}").hostname or ""
        if is_ip(hostname):
            return ""
        parts = hostname.rstrip(".").split(".")
        tld = extract_tld_fallback(url)
        tld_parts = len(tld.split("."))
        if len(parts) > tld_parts:
            return parts[-(tld_parts + 1)]
        return ""
    except Exception:
        return ""


def extract_subdomain(url: str) -> str:
    if HAS_TLDEXTRACT:
        ext = tldextract.extract(url)
        return ext.subdomain.lower() if ext.subdomain else ""
    try:
        hostname = urlparse(url if "://" in url else f"http://{url}").hostname or ""
        if is_ip(hostname):
            return ""
        parts = hostname.rstrip(".").split(".")
        tld = extract_tld_fallback(url)
        tld_parts = len(tld.split("."))
        if len(parts) > tld_parts + 1:
            return ".".join(parts[:-(tld_parts + 1)])
        return ""
    except Exception:
        return ""


# Build TLD frequency table from legitimate URLs
if os.path.exists(TOP1M_PATH):
    tld_corpus = top1m["domain"].dropna().apply(lambda d: extract_tld(d))
else:
    tld_corpus = combined[combined["label"] == 0]["url_clean"].apply(extract_tld)

tld_counts = Counter(tld_corpus)
total_tld_count = sum(tld_counts.values())

TLD_LEGIT_PROB = {}
for tld, count in tld_counts.items():
    if tld:
        TLD_LEGIT_PROB[tld] = count / total_tld_count

print(f"       Unique TLDs: {len(TLD_LEGIT_PROB):,}")
print(f"       Top-5 TLDs: {sorted(TLD_LEGIT_PROB.items(), key=lambda x: -x[1])[:5]}")

# ---------- 2c. Sensitive words list (Chiew 2019, feature #25) ----------
# "Counts the number of sensitive words (i.e., 'secure', 'account', 'webscr',
#  'login', 'ebayisapi', 'signin', 'banking', 'confirm') in webpage URL"

SENSITIVE_WORDS = [
    "secure", "account", "webscr", "login", "ebayisapi", "signin",
    "banking", "confirm", "update", "verify", "password", "suspend",
    "paypal", "authenticate", "wallet", "credential"
]
# Extended from Chiew's base list with common phishing vocabulary

print(f"\n  [2c] Sensitive words list: {len(SENSITIVE_WORDS)} words")
print(f"       Words: {SENSITIVE_WORDS}")

# ═══════════════════════════════════════════════════════════════
# PHASE 3 — FEATURE EXTRACTION ENGINE (30 features)
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 3: Extracting 30 URL-structural features")
print("=" * 80)


def shannon_entropy(s: str) -> float:
    """Shannon entropy: E = -Σ(p_i × log₂(p_i))
    Tamal et al. (2024) F40: 'Representing the Shannon entropy of the URL.
    It is a continuous feature calculated based on the probabilities of each
    character in the URL.'
    """
    if not s:
        return 0.0
    freq = Counter(s)
    length = len(s)
    return -sum((c / length) * math.log2(c / length) for c in freq.values() if c > 0)


def url_char_prob(url: str) -> float:
    """PhiUSIIL §3.1.4 Eq. (1):
    URLCharProb = Σ prob(URL[char_i]) / n
    where n = count of a-z and 0-9 characters in URL,
    prob() = pre-computed from legitimate URL corpus.
    """
    url_lower = url.lower()
    alnum_chars = [c for c in url_lower if c.isalnum()]
    n = len(alnum_chars)
    if n == 0:
        return 0.0
    return sum(CHAR_PROB_TABLE.get(c, 0.0) for c in alnum_chars) / n


def char_continuation_rate(url: str) -> float:
    """PhiUSIIL §3.1.4:
    'To find the CharContinuationRate of a URL, we identified the longest
    alphabet, digit, and special character sequences and total their length.
    Further, the total length of these sequences is divided by the totaled
    length of the URL.'
    CCR = (max_alpha_seq + max_digit_seq + max_special_seq) / len(URL)
    """
    if not url:
        return 0.0

    max_alpha = max_digit = max_special = 0
    cur_alpha = cur_digit = cur_special = 0

    for c in url:
        if c.isalpha():
            cur_alpha += 1
            max_alpha = max(max_alpha, cur_alpha)
            cur_digit = cur_special = 0
        elif c.isdigit():
            cur_digit += 1
            max_digit = max(max_digit, cur_digit)
            cur_alpha = cur_special = 0
        else:
            cur_special += 1
            max_special = max(max_special, cur_special)
            cur_alpha = cur_digit = 0

    length = len(url)
    return (max_alpha + max_digit + max_special) / length if length > 0 else 0.0


# Repeated digits — Tamal (2024) F3: "A Boolean feature that denotes whether
# the URL has repeated digits (e.g., 2232)"
REPEATED_DIGITS_PATTERN = re.compile(r"(\d)\1{2,}")

# Suspicious file extensions — LegitPhish: "Suspicious file type used"
SUSPICIOUS_EXTENSIONS = {
    ".exe", ".zip", ".rar", ".scr", ".bat", ".cmd", ".msi", ".dll",
    ".vbs", ".js", ".jar", ".ps1", ".wsf", ".lnk", ".7z", ".cab"
}


def extract_features(url: str) -> dict:
    """Extract all 30 low-latency features from a single URL string.
    
    Every feature is cited to its source paper and extraction method
    matches the paper's specification exactly.
    """
    features = {}

    # Parse URL components once
    url_str = url.strip()
    parsed = urlparse(url_str if "://" in url_str else f"http://{url_str}")
    hostname = (parsed.hostname or "").lower()
    path = parsed.path or ""
    query = parsed.query or ""
    fragment = parsed.fragment or ""
    scheme = (parsed.scheme or "").lower()

    # Domain decomposition
    tld = extract_tld(url_str)
    domain = extract_domain(url_str)
    subdomain = extract_subdomain(url_str)
    subdomain_parts = [p for p in subdomain.split(".") if p] if subdomain else []

    # ─── TIER 1: Strongest features (Rank 1–10) ───────────────

    # F01: url_length — ALL 5 PAPERS
    # Mohammad (2012): "if the length of the URL is greater than or equal 54
    # characters then the URL classified as phishing" (48.8% of dataset).
    features["url_length"] = len(url_str)

    # F02: num_dots — ALL 5 PAPERS
    # Chiew (2019): feature #1 NumDots. Tamal (2024): F2.
    features["num_dots"] = url_str.count(".")

    # F03: num_subdomains — ALL 5 PAPERS
    # Mohammad (2012): "if the number of dots is equal to three then the URL
    # is classified as 'suspicious' since it has one sub domain" (44.4%).
    # Tamal (2024) F24: number_of_subdomains.
    features["num_subdomains"] = len(subdomain_parts)

    # F04: has_ip_address — 4/5 PAPERS
    # Mohammad (2012): "If IP address is used instead of domain name in the URL
    # ...the user can almost be sure someone is trying to steal his personal
    # information" (22.8%). Check between // and /.
    features["has_ip_address"] = 1 if is_ip(hostname) else 0

    # F05: num_hyphens_url — 5/5 PAPERS, HEFS BASELINE
    # Chiew (2019): NumDash — SELECTED AS HEFS BASELINE FEATURE.
    # Mohammad (2012): "Dash is rarely used in legitimate URL" (26.4%).
    features["num_hyphens_url"] = url_str.count("-")

    # F06: num_hyphens_hostname — HEFS BASELINE COMPLEMENT
    # Chiew (2019): feature #6 NumDashInHostname.
    features["num_hyphens_hostname"] = hostname.count("-")

    # F07: https_flag — 4/5 PAPERS
    # Mohammad (2012): "92.8% of the dataset does not support HTTPS or use a
    # fake https." PhiUSIIL: "A URL using the http protocol is highly likely
    # to be a phishing URL."
    features["https_flag"] = 1 if scheme == "https" else 0

    # F08: entropy_url — 3/5 PAPERS
    # Tamal (2024) F40: E = -Σ(p_i × log₂(p_i)) where p_i = probability of
    # each character in the URL.
    features["entropy_url"] = round(shannon_entropy(url_str), 6)

    # F09: num_numeric_chars — 4/5 PAPERS, HEFS BASELINE
    # Chiew (2019): NumNumericChars — SELECTED AS HEFS BASELINE FEATURE.
    # PhiUSIIL: "A large number of digits increases the possibility."
    features["num_numeric_chars"] = sum(c.isdigit() for c in url_str)

    # F10: num_sensitive_words — HEFS BASELINE
    # Chiew (2019) feature #25: "Counts the number of sensitive words (i.e.,
    # 'secure', 'account', 'webscr', 'login', 'ebayisapi', 'signin',
    # 'banking', 'confirm') in webpage URL."
    url_lower = url_str.lower()
    features["num_sensitive_words"] = sum(url_lower.count(w) for w in SENSITIVE_WORDS)

    # ─── TIER 2: Strong features (Rank 11–20) ─────────────────

    # F11: hostname_length — 3/5 PAPERS
    # Chiew (2019): feature #21 HostnameLength. LegitPhish: domain_name_length.
    features["hostname_length"] = len(hostname)

    # F12: path_length — 3/5 PAPERS
    # Chiew (2019): feature #22 PathLength. Tamal (2024): F36. LegitPhish.
    features["path_length"] = len(path)

    # F13: url_char_prob — PhiUSIIL Eq. (1)
    # "most of the characters such as a, c, e, o, r, and t are more frequently
    # used in legitimate URLs, while alphabets such as b, f, q, v, w, x, y, z,
    # and all the digits are more frequent in phishing URLs."
    features["url_char_prob"] = round(url_char_prob(url_str), 8)

    # F14: char_continuation_rate — PhiUSIIL §3.1.4
    # "URLs such as www.a1b2c3.com or www.a1_b-_2.com get lower
    # CharContinuationRate" — lower rate = more randomized = suspicious.
    features["char_continuation_rate"] = round(char_continuation_rate(url_str), 6)

    # F15: tld_legit_prob — PhiUSIIL §3.1.4
    # "A higher TLDLegitimateProb of a URL may indicate a legitimate URL,
    # and a lower TLDLegitimateProb value may help identify phishing URLs."
    features["tld_legit_prob"] = round(TLD_LEGIT_PROB.get(tld, 0.0), 8)

    # F16: entropy_domain — Tamal (2024) F41
    # "Representing the Shannon entropy of the domain."
    features["entropy_domain"] = round(shannon_entropy(domain), 6)

    # F17: num_query_params — 4/5 PAPERS
    # Chiew (2019): #11 NumQueryComponents, #12 NumAmpersand.
    # Count literal & delimiters to capture repetitive obfuscation like
    # ?id=1&id=2&id=3 (parse_qs would group these as 1 key).
    features["num_query_params"] = len(query.split("&")) if query else 0

    # F18: num_special_chars — 2/5 PAPERS
    # Tamal (2024) F5: "Number of special characters (e.g., !, #, $, %, &, ~)."
    # PhiUSIIL: NoOfObfuscatedChar.
    features["num_special_chars"] = sum(
        1 for c in url_str if c in "!@#$%^&*~`|\\<>{}"
    )

    # F19: at_symbol_present — 4/5 PAPERS
    # Mohammad (2012): "browser might ignore everything prior the @ symbol"
    # (3.6% frequency — rare but highly indicative when present).
    features["at_symbol_present"] = 1 if "@" in url_str else 0

    # F20: pct_numeric_chars — LegitPhish: percentage_numeric_chars
    # Ratio-normalized version of F09.
    features["pct_numeric_chars"] = round(
        features["num_numeric_chars"] / max(len(url_str), 1), 6
    )

    # ─── TIER 3: Useful features (Rank 21–30) ─────────────────

    # F21: suspicious_file_ext — LegitPhish
    # "Suspicious file type used (0/1)"
    path_lower = path.lower()
    features["suspicious_file_ext"] = 1 if any(
        path_lower.endswith(ext) for ext in SUSPICIOUS_EXTENSIONS
    ) else 0

    # F22: path_depth — 2/5 PAPERS
    # Chiew (2019): feature #3 PathLevel. Tamal (2024): F8.
    # Count '/' in path, excluding the leading slash.
    features["path_depth"] = max(path.count("/") - 1, 0)

    # F23: num_underscores — 2/5 PAPERS
    # Chiew (2019): feature #9 NumUnderscore. Tamal (2024): F7.
    features["num_underscores"] = url_str.count("_")

    # F24: double_slash_in_path — 1/5 PAPERS
    # Chiew (2019): feature #24 DoubleSlashInPath.
    # Check if '//' appears in the path (after the protocol).
    features["double_slash_in_path"] = 1 if "//" in path else 0

    # F25: query_length — 2/5 PAPERS
    # Chiew (2019): feature #23 QueryLength.
    features["query_length"] = len(query)

    # F26: has_fragment — 2/5 PAPERS
    # Tamal (2024): F38 having_fragment. Chiew (2019): #13 NumHash.
    features["has_fragment"] = 1 if fragment else 0

    # F27: has_repeated_digits — Tamal (2024) NOVEL
    # F3: "A Boolean feature that denotes whether the URL has repeated
    # digits (e.g., 2232)"
    features["has_repeated_digits"] = 1 if REPEATED_DIGITS_PATTERN.search(url_str) else 0

    # F28: avg_subdomain_length — Tamal (2024) F27 NOVEL
    # "Representing the average length of the subdomains in the URL."
    features["avg_subdomain_length"] = round(
        sum(len(p) for p in subdomain_parts) / max(len(subdomain_parts), 1), 4
    )

    # F29: tld_length — LegitPhish
    # Length of the top-level domain string.
    features["tld_length"] = len(tld)

    # F30: token_count — LegitPhish
    # Split URL by common delimiters and count tokens.
    tokens = re.split(r"[/\?\&\=\-\_\.\:\@\#\+\~\%]", url_str)
    features["token_count"] = len([t for t in tokens if t])

    return features


# ─── Run extraction ───────────────────────────────────────────

print(f"\n  Extracting features from {len(combined):,} URLs...")
print(f"  (This may take 5–15 minutes depending on dataset size)")

# Process in chunks to show progress
CHUNK_SIZE = 10000
all_features = []
urls = combined["url_clean"].tolist()

for i in range(0, len(urls), CHUNK_SIZE):
    chunk = urls[i:i + CHUNK_SIZE]
    chunk_features = [extract_features(u) for u in chunk]
    all_features.extend(chunk_features)
    processed = min(i + CHUNK_SIZE, len(urls))
    pct = processed / len(urls) * 100
    print(f"    Processed {processed:>8,} / {len(urls):,} ({pct:5.1f}%)")

features_df = pd.DataFrame(all_features)
print(f"\n  Feature matrix shape: {features_df.shape}")

# ═══════════════════════════════════════════════════════════════
# PHASE 4 — ASSEMBLE FINAL DATASET
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 4: Assembling final dataset")
print("=" * 80)

# Combine: url + label + source + 30 features
final = pd.concat([
    combined[["url_clean", "label", "source"]].reset_index(drop=True),
    features_df.reset_index(drop=True)
], axis=1)
final = final.rename(columns={"url_clean": "url"})

print(f"  Final shape: {final.shape[0]:,} rows × {final.shape[1]} cols")

# ═══════════════════════════════════════════════════════════════
# PHASE 5 — DATA INTEGRITY VALIDATION
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 5: Data integrity validation")
print("=" * 80)

# Check for nulls
null_report = final.isnull().sum()
null_cols = null_report[null_report > 0]
if len(null_cols) > 0:
    print("  ⚠ Columns with nulls:")
    for col, cnt in null_cols.items():
        print(f"    {col}: {cnt:,} nulls ({cnt/len(final)*100:.2f}%)")
    # Fill numeric nulls with 0 (safe default for counts/flags)
    feature_cols = [c for c in final.columns if c not in ["url", "label", "source"]]
    final[feature_cols] = final[feature_cols].fillna(0)
    print("    → Filled with 0")
else:
    print("  ✓ No null values")

# Check for infinite values
feature_cols = [c for c in final.columns if c not in ["url", "label", "source"]]
inf_mask = final[feature_cols].apply(lambda s: np.isinf(s) if np.issubdtype(s.dtype, np.floating) else pd.Series(False, index=s.index))
inf_count = inf_mask.sum().sum()
if inf_count > 0:
    print(f"  ⚠ {inf_count} infinite values found — replacing with 0")
    final[feature_cols] = final[feature_cols].replace([np.inf, -np.inf], 0)
else:
    print("  ✓ No infinite values")

# Verify label integrity
assert final["label"].isin([0, 1]).all(), "Labels must be 0 or 1!"
print("  ✓ Labels are all 0 or 1")

# Verify no duplicate URLs
assert final["url"].duplicated().sum() == 0, "Duplicate URLs found after dedup!"
print("  ✓ No duplicate URLs")

# IQR-based outlier report (per Tamal et al. recommendation, but don't remove — just flag)
print("\n  IQR Outlier Report (for reference, not removed):")
for col in ["url_length", "hostname_length", "path_length"]:
    q1 = final[col].quantile(0.25)
    q3 = final[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = ((final[col] < lower) | (final[col] > upper)).sum()
    print(f"    {col}: Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f}, "
          f"bounds=[{lower:.0f}, {upper:.0f}], outliers={outliers:,}")

# Feature statistics summary
print("\n  Feature statistics:")
print(final[feature_cols].describe().T[["mean", "std", "min", "50%", "max"]].to_string(
    float_format=lambda x: f"{x:.4f}" if abs(x) < 1e6 else f"{x:.2e}"
))

# ═══════════════════════════════════════════════════════════════
# PHASE 6 — SAVE OUTPUT
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 6: Saving final dataset")
print("=" * 80)

final.to_csv(OUTPUT_PATH, index=False)
file_size = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"  ✓ Saved to: {OUTPUT_PATH}")
print(f"  ✓ File size: {file_size:.1f} MB")
print(f"  ✓ Shape: {final.shape[0]:,} rows × {final.shape[1]} cols")
print(f"  ✓ Columns: {list(final.columns)}")

# Final summary
print("\n" + "=" * 80)
print("PIPELINE COMPLETE — FINAL SUMMARY")
print("=" * 80)
print(f"""
  Dataset:    {OUTPUT_PATH}
  Rows:       {final.shape[0]:,}
  Features:   {len(feature_cols)} (+ url, label, source)
  Classes:    0=legitimate ({(final['label']==0).sum():,}), 1=phishing ({(final['label']==1).sum():,})
  Balance:    {(final['label']==0).sum()/len(final)*100:.1f}% legit / {(final['label']==1).sum()/len(final)*100:.1f}% phish
  Sources:    {dict(final['source'].value_counts())}
  
  Feature list (30 features, ordered by tier):
  ┌─────────────────────────────────────────────────────────────┐
  │ TIER 1 (F01-F10): url_length, num_dots, num_subdomains,     │ 
  │   has_ip_address, num_hyphens_url, num_hyphens_hostname,    │
  │   https_flag, entropy_url, num_numeric_chars,               │
  │   num_sensitive_words                                       │
  │ TIER 2 (F11-F20): hostname_length, path_length,             │
  │   url_char_prob, char_continuation_rate, tld_legit_prob,    │
  │   entropy_domain, num_query_params, num_special_chars,      │
  │   at_symbol_present, pct_numeric_chars                      │
  │ TIER 3 (F21-F30): suspicious_file_ext, path_depth,          │
  │   num_underscores, double_slash_in_path, query_length,      │
  │   has_fragment, has_repeated_digits, avg_subdomain_length,  │
  │   tld_length, token_count                                   │
  └─────────────────────────────────────────────────────────────┘
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 5.7 MB/s eta 0:00:00
PHASE 0: Loading datasets and unifying to (url, label)
  DS2 (PhiUSIIL) loaded: 235,795 rows × 56 cols
  DS2 after strip: 235,795 rows — label encoding: 0=legit, 1=phish
  DS3 (LegitPhish) loaded: 101,219 rows × 18 cols
  DS3 after strip: 101,219 rows — label encoding: 0=legit, 1=phish

  Combined: 337,014 rows

PHASE 1: Data quality checks & deduplication
  Dropped 1 rows with null url/label
  Dropped 1 rows with malformed URLs (<10 chars or no dots)
  Deduplicated: removed 37,706 duplicate URLs
  Remaining: 299,306 unique URLs

  Class distribution:
    Legitimate (0):    135,295 (45.2%)
    Phishing   (1):    164,011 (54.8%)
    Source distribution:
source
PhiUSIIL      235370
LegitPhish     63936

PHASE 2: Building lookup tables for derived features

  [2a] Building URLCharProb lookup table...
       Using Cisco Umbrella top-1M from /kaggle/input/datasets/saifeldenismail/top-1m/top-1m.csv
       Corpus 

In [2]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║  CyberSiren — Model Selection & Benchmarking Pipeline              ║
║  Tests 10 models on PullDD dataset (299,306 URLs × 30 features)   ║
║  Paper-driven model selection + one bleeding-edge approach         ║
╚══════════════════════════════════════════════════════════════════════╝

Model rationale from literature:
────────────────────────────────
 • Random Forest    — Chiew (2019): "consistently outperform all remaining
                      classifiers by scoring above 95% accuracy."
                      PhiUSIIL Table 3: 99.982% accuracy.
 • XGBoost          — PhiUSIIL Table 3: 99.993% (highest of all tested).
 • LightGBM         — PhiUSIIL Table 3: 99.99% (fastest boosting method).
 • CatBoost         — PhiUSIIL Table 3: 99.987%.
 • Logistic Reg.    — PhiUSIIL Table 3: 99.654% (strong linear baseline).
 • Decision Tree    — Chiew (2019): C4.5 at 94.37%. Simple interpretable.
 • Extra Trees      — Randomized RF variant, often better generalization.
 • AdaBoost         — PhiUSIIL Table 3: 99.981%.
 • Voting Ensemble  — PhiUSIIL §4.2.2: "ensemble of multiple ML algorithms
                      has shown better classification." Their BestSecurity
                      uses 2/3 consensus voting.
 • Stacking         — PhiUSIIL Table 3: 99.979%. "Bleeding edge" approach:
                      stack the top-3 base models with LR meta-learner.
"""

import pandas as pd
import numpy as np
import time
import warnings
import os
import json
from collections import OrderedDict

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve,
    average_precision_score, log_loss, balanced_accuracy_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
    VotingClassifier, StackingClassifier, GradientBoostingClassifier
)

# Boosting libraries
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠ xgboost not available — skipping XGBoost")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("⚠ lightgbm not available — skipping LightGBM")

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("⚠ catboost not available — skipping CatBoost")

warnings.filterwarnings("ignore")
np.random.seed(42)

# ═══════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════

DATASET_PATH = "/kaggle/input/datasets/saifeldenismail/pulldd-phishing-url-low-latency-detection-dataset/cybersiren_lowlatency_dataset(2).csv"
RESULTS_PATH = "/kaggle/working/cybersiren_benchmark_results.csv"

# Split ratios — standard 70/15/15
# Chiew (2019) §5.2: "70% of the data is utilised for training while 30%
# is reserved for testing purpose." We carve out validation from that 30%.
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

RANDOM_STATE = 42

# ─── Production Profiles ───────────────────────────────────────────────
# PATCH 3: Corrected comment. Original said "Aggressively suppress FPs on
# legit blogs/articles" — but the actual error profile is 61 FNs vs 19 FPs,
# so raising the threshold makes the dominant problem (FNs) worse for any
# borderline case near 0.50.
#
# TRADEOFF: raising this threshold reduces FPs but increases FNs.
# Current error profile: 61 FNs vs 19 FPs (FNR >> FPR).
# At threshold=0.85, borderline phishing near 0.50 gets reclassified as safe.
# The dominant FN cluster (prob < 0.30, bare-domain combosquats) is completely
# unaffected by threshold adjustment — those require a similarity index, not
# threshold tuning (see Phase 8 for details).
PRODUCTION_THRESHOLD = 0.85

# ═══════════════════════════════════════════════════════════════
# PHASE 0 — LOAD & PREPARE DATA
# ═══════════════════════════════════════════════════════════════

print("=" * 80)
print("PHASE 0: Loading PullDD dataset")
print("=" * 80)

df = pd.read_csv(DATASET_PATH, low_memory=False)
print(f"  Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

# Identify feature columns (everything except url, label, source)
META_COLS = ["url", "label", "source"]
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]
print(f"  Features: {len(FEATURE_COLS)}")
print(f"  Label distribution: {dict(df['label'].value_counts())}")

# ─── Data quality pre-check ───
print("\n  Pre-check:")
zero_var_cols = [c for c in FEATURE_COLS if df[c].std() == 0]
if zero_var_cols:
    print(f"  ⚠ Dropping zero-variance features: {zero_var_cols}")
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in zero_var_cols]
    print(f"  Features after drop: {len(FEATURE_COLS)}")

# ─── PATCH 2: Feature Schema Pruning (importance-based, post-hoc) ──────
# has_ip_address and double_slash_in_path contributed 0 splits in the
# champion LightGBM model (confirmed in ML-SPEC-v1.1 §5, feature importance
# table). Removing them reduces dimensionality without measurable accuracy cost.
#
# NOTE: this is NOT HEFS/CDF-g. HEFS applies the CDF-g algorithm across
# data-perturbation and function-perturbation cycles to automatically locate
# the optimal cutoff rank from the distribution shape of filter measure values
# (Chiew 2019, §4.2). What follows is a hardcoded removal based on a prior
# training run's importance measurement — valid, but a different procedure.
obsolete_cols = ["has_ip_address", "double_slash_in_path"]
cols_to_prune = [c for c in obsolete_cols if c in FEATURE_COLS]
if cols_to_prune:
    print(f"  ⚠ Pruning zero-importance features: {cols_to_prune}")
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in cols_to_prune]
    print(f"  Features after pruning: {len(FEATURE_COLS)}")

# Check for any remaining nulls/infs
null_count = df[FEATURE_COLS].isnull().sum().sum()
inf_count  = np.isinf(df[FEATURE_COLS].select_dtypes(include=[np.number])).sum().sum()
print(f"  Nulls: {null_count}, Infs: {inf_count}")
assert null_count == 0 and inf_count == 0, "Data quality issue!"
print("  ✓ Data quality passed")

X = df[FEATURE_COLS]  # Keep as DataFrame — preserves feature names for LightGBM/XGBoost
y = df["label"]       # Keep as Series
feature_names = FEATURE_COLS

print(f"\n  X shape: {X.shape}")
print(f"  y shape: {y.shape}, pos_rate: {y.mean():.4f}")

# ═══════════════════════════════════════════════════════════════
# PHASE 1 — TRAIN / VAL / TEST SPLIT
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 1: Stratified train/val/test split (70/15/15)")
print("=" * 80)

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=(VAL_RATIO + TEST_RATIO),
    stratify=y, random_state=RANDOM_STATE
)

# Second split: split temp 50/50 into val and test (each 15% of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5,
    stratify=y_temp, random_state=RANDOM_STATE
)

print(f"  Train: {X_train.shape[0]:,} ({X_train.shape[0]/len(X)*100:.1f}%) "
      f"— pos_rate: {y_train.mean():.4f}")
print(f"  Val:   {X_val.shape[0]:,} ({X_val.shape[0]/len(X)*100:.1f}%) "
      f"— pos_rate: {y_val.mean():.4f}")
print(f"  Test:  {X_test.shape[0]:,} ({X_test.shape[0]/len(X)*100:.1f}%) "
      f"— pos_rate: {y_test.mean():.4f}")

# Scale for models that benefit from it (LR, SVM-based).
# Tree-based models are invariant to scaling but it doesn't hurt.
# set_output("pandas") keeps DataFrame format → eliminates LightGBM
# warnings about missing feature names inside StackingClassifier CV.
scaler = StandardScaler().set_output(transform="pandas")
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ═══════════════════════════════════════════════════════════════
# PHASE 2 — MODEL DEFINITIONS
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 2: Defining models")
print("=" * 80)

models = OrderedDict()

# ─── 1. Logistic Regression (baseline) ────────────────────────
# PhiUSIIL Table 3: 99.654% accuracy. Strong linear baseline.
models["LogisticRegression"] = {
    "model": LogisticRegression(
        max_iter=1000, C=1.0, solver="lbfgs", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "needs_scaling": True,
    "paper": "PhiUSIIL Table 3: 99.654%",
}

# ─── 2. Decision Tree ─────────────────────────────────────────
# Chiew (2019): C4.5 at 94.37%. PhiUSIIL Table 3: 99.866%.
models["DecisionTree"] = {
    "model": DecisionTreeClassifier(
        random_state=RANDOM_STATE, max_depth=None, min_samples_leaf=5
    ),
    "needs_scaling": False,
    "paper": "Chiew (2019): C4.5 at 94.37%",
}

# ─── 3. Random Forest — PAPER CONSENSUS BEST ─────────────────
# Chiew (2019) §5.3: "Random Forest appears to consistently outperform
# all remaining classifiers by scoring above 95% accuracy."
# PhiUSIIL Table 3: 99.982%.
models["RandomForest"] = {
    "model": RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "needs_scaling": False,
    "paper": "Chiew (2019): consensus best. PhiUSIIL: 99.982%",
}

# ─── 4. Extra Trees ───────────────────────────────────────────
# Same family as RF but splits are chosen randomly → lower variance,
# often better generalization on noisy URL features.
models["ExtraTrees"] = {
    "model": ExtraTreesClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "needs_scaling": False,
    "paper": "RF variant — tested for generalization comparison",
}

# ─── 5. AdaBoost ──────────────────────────────────────────────
# PhiUSIIL Table 3: 99.981%. Sharma & Singh (2022) used AdaBoost.
models["AdaBoost"] = {
    "model": AdaBoostClassifier(
        n_estimators=200, learning_rate=0.1, random_state=RANDOM_STATE
    ),
    "needs_scaling": False,
    "paper": "PhiUSIIL Table 3: 99.981%",
}

# ─── 6. XGBoost — PhiUSIIL TOP PERFORMER ─────────────────────
# PhiUSIIL Table 3: 99.993% accuracy (highest of all algorithms tested).
if HAS_XGB:
    # use_label_encoder was removed in xgboost >=2.0. Build params safely.
    _xgb_params = dict(
        n_estimators=300, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        gamma=0.1, reg_alpha=0.01, reg_lambda=1.0,
        eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    try:
        XGBClassifier(use_label_encoder=False, n_estimators=1).get_params()
        _xgb_params["use_label_encoder"] = False
    except TypeError:
        pass  # parameter removed in this version

    models["XGBoost"] = {
        "model": XGBClassifier(**_xgb_params),
        "needs_scaling": False,
        "paper": "PhiUSIIL Table 3: 99.993% — HIGHEST",
    }

# ─── 7. LightGBM — FASTEST BOOSTING ──────────────────────────
# PhiUSIIL Table 3: 99.99% accuracy, training time = 1 (fastest).
if HAS_LGBM:
    models["LightGBM"] = {
        "model": LGBMClassifier(
            n_estimators=300, max_depth=8, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
            reg_alpha=0.01, reg_lambda=1.0, num_leaves=63,
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
        ),
        "needs_scaling": False,
        "paper": "PhiUSIIL Table 3: 99.99%, fastest training",
    }

# ─── 8. CatBoost ─────────────────────────────────────────────
# PhiUSIIL Table 3: 99.987%.
if HAS_CATBOOST:
    models["CatBoost"] = {
        "model": CatBoostClassifier(
            iterations=300, depth=8, learning_rate=0.1,
            l2_leaf_reg=3.0, random_seed=RANDOM_STATE,
            verbose=0, thread_count=-1
        ),
        "needs_scaling": False,
        "paper": "PhiUSIIL Table 3: 99.987%",
    }

# ─── 9. Soft Voting Ensemble ─────────────────────────────────
# PhiUSIIL §4.2.2: "The ensemble of multiple machine learning algorithms
# has shown better classification in various studies."
# PhiUSIIL Table 3: Vote_Soft 99.817%.
# We ensemble the top-3 tree-based models.
vote_estimators = [("rf", models["RandomForest"]["model"])]
if HAS_XGB:
    vote_estimators.append(("xgb", models["XGBoost"]["model"]))
if HAS_LGBM:
    vote_estimators.append(("lgbm", models["LightGBM"]["model"]))

if len(vote_estimators) >= 2:
    models["VotingEnsemble"] = {
        "model": VotingClassifier(
            estimators=vote_estimators, voting="soft", n_jobs=-1
        ),
        "needs_scaling": False,
        "paper": "PhiUSIIL §4.2.2: multi-model consensus voting",
    }

# ─── 10. Stacking Classifier — BLEEDING EDGE ─────────────────
# PhiUSIIL Table 3: StackingClassifier 99.979%.
# Rationale: Stack the top-3 diverse base learners with a logistic
# regression meta-learner. This combines the complementary strengths:
# - RF captures feature interactions via bagging
# - XGBoost captures residual patterns via boosting
# - LightGBM provides a different gradient perspective
# The LR meta-learner learns optimal weighting of their predictions.
stack_estimators = [("rf", RandomForestClassifier(
    n_estimators=200, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
))]
if HAS_XGB:
    _xgb_stack = dict(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    try:
        XGBClassifier(use_label_encoder=False, n_estimators=1).get_params()
        _xgb_stack["use_label_encoder"] = False
    except TypeError:
        pass
    stack_estimators.append(("xgb", XGBClassifier(**_xgb_stack)))
if HAS_LGBM:
    stack_estimators.append(("lgbm", LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )))

if len(stack_estimators) >= 2:
    models["StackingEnsemble"] = {
        "model": StackingClassifier(
            estimators=stack_estimators,
            final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            cv=5, stack_method="predict_proba", n_jobs=-1, passthrough=False
        ),
        "needs_scaling": False,
        "paper": "PhiUSIIL Table 3: 99.979%. Bleeding edge: diverse base + LR meta",
    }

print(f"  Models defined: {len(models)}")
for name, cfg in models.items():
    print(f"    • {name:25s} — {cfg['paper']}")

# ═══════════════════════════════════════════════════════════════
# PHASE 3 — TRAINING & EVALUATION
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 3: Training & evaluation on validation set")
print("=" * 80)

results = []

for name, cfg in models.items():
    print(f"\n{'─'*70}")
    print(f"  Training: {name}")
    print(f"  Paper ref: {cfg['paper']}")

    model    = cfg["model"]
    use_scaled = cfg["needs_scaling"]

    Xt = X_train_sc if use_scaled else X_train
    Xv = X_val_sc   if use_scaled else X_val

    # Train
    t0 = time.time()
    model.fit(Xt, y_train)
    train_time = time.time() - t0

    # Predict on validation
    t0 = time.time()
    y_pred = model.predict(Xv)
    pred_time = time.time() - t0

    # Probabilities for AUC
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(Xv)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = model.decision_function(Xv)
    else:
        y_prob = y_pred.astype(float)

    # ─── PATCH 1: Spec-correct enrichment routing band ──────────────────
    # ML-SPEC-v1.1 §8.2 defines two enrichment tiers:
    #   UNCERTAIN  : [0.30, 0.50) — route to WHOIS/SSL/DNS enrichment
    #   SUSPICIOUS : [0.50, 0.85) — route to WHOIS/SSL/DNS enrichment
    # Combined enrichment band is therefore [0.30, 0.85).
    # Prior version used (0.3, 0.7) which left the SUSPICIOUS tier
    # (0.70–0.85) stranded and understated the routed_pct metric.
    uncertainty_mask = (y_prob >= 0.30) & (y_prob < 0.85)
    routed_pct = (uncertainty_mask.sum() / len(y_prob)) * 100

    # ─── Calibrated threshold evaluation (diagnostic only) ──────────────
    y_pred_calibrated = (y_prob >= PRODUCTION_THRESHOLD).astype(int)
    tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_val, y_pred_calibrated).ravel()
    fpr_calibrated = fp_c / (fp_c + tn_c) if (fp_c + tn_c) > 0 else 0

    # Metrics
    acc     = accuracy_score(y_val, y_pred)
    bal_acc = balanced_accuracy_score(y_val, y_pred)
    prec    = precision_score(y_val, y_pred)
    rec     = recall_score(y_val, y_pred)
    f1      = f1_score(y_val, y_pred)
    mcc     = matthews_corrcoef(y_val, y_pred)
    auc     = roc_auc_score(y_val, y_prob)
    ap      = average_precision_score(y_val, y_prob)
    ll      = log_loss(y_val, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    # Phishing-specific metrics
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # false positive rate
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # false negative rate

    # Batch latency (vectorized — throughput metric)
    batch_latency_us = (pred_time / len(y_val)) * 1e6

    # Single-inference latency (production API reality — no vectorization)
    sample_size = min(1000, len(Xv))
    Xv_sample = Xv.iloc[:sample_size] if hasattr(Xv, "iloc") else Xv[:sample_size]
    t0_single = time.time()
    for i in range(sample_size):
        row = Xv_sample.iloc[[i]] if hasattr(Xv_sample, "iloc") else Xv_sample[[i]]
        _ = model.predict(row)
    single_latency_us = ((time.time() - t0_single) / sample_size) * 1e6

    result = {
        "model": name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "mcc": mcc,
        "auc_roc": auc,
        "avg_precision": ap,
        "log_loss": ll,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "fpr": fpr,
        "fnr": fnr,
        "calibrated_fpr": fpr_calibrated,
        "routed_pct": routed_pct,
        "train_time_s": train_time,
        "pred_time_s": pred_time,
        "batch_latency_us": batch_latency_us,
        "single_latency_us": single_latency_us,
    }
    results.append(result)

    print(f"  ✓ Done in {train_time:.1f}s")
    print(f"    Acc={acc:.5f}  F1={f1:.5f}  MCC={mcc:.5f}  "
          f"AUC={auc:.5f}  LogLoss={ll:.5f}")
    print(f"    TP={tp:,} TN={tn:,} FP={fp:,} FN={fn:,}")
    print(f"    FPR={fpr:.5f} FNR={fnr:.5f}")
    print(f"    FPR (threshold={PRODUCTION_THRESHOLD}): {fpr_calibrated:.5f}  "
          f"| Enrichment-routed: {routed_pct:.2f}% of URLs")
    print(f"    Batch={batch_latency_us:.1f}μs/URL  Single={single_latency_us:.1f}μs/URL")

# ═══════════════════════════════════════════════════════════════
# PHASE 4 — VALIDATION LEADERBOARD
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 4: Validation leaderboard (ranked by MCC)")
print("=" * 80)

res_df = pd.DataFrame(results)
res_df = res_df.sort_values("mcc", ascending=False).reset_index(drop=True)
res_df.index = res_df.index + 1  # 1-indexed rank
res_df.index.name = "rank"

# Display key metrics
display_cols = ["model", "accuracy", "precision", "recall", "f1_score",
                "mcc", "auc_roc", "log_loss", "fpr", "fnr",
                "calibrated_fpr", "routed_pct",
                "train_time_s", "batch_latency_us", "single_latency_us"]
print("\n" + res_df[display_cols].to_string(
    float_format=lambda x: f"{x:.5f}" if isinstance(x, float) and abs(x) < 100 else f"{x:.1f}"
))

# Why MCC for ranking:
print("""
  ┌─────────────────────────────────────────────────────────────────┐
  │ Ranking by MCC (Matthews Correlation Coefficient) because:      │
  │ • Unlike accuracy, MCC is balanced even with class imbalance   │
  │ • Unlike F1, MCC accounts for all four confusion matrix cells  │
  │ • Range [-1, +1] where +1 = perfect, 0 = random, -1 = inverse │
  │ • PhiUSIIL Eq. (7) uses MCC as a primary evaluation metric     │
  └─────────────────────────────────────────────────────────────────┘
""")

# ═══════════════════════════════════════════════════════════════
# PHASE 5 — FINAL EVALUATION ON HELD-OUT TEST SET
# ═══════════════════════════════════════════════════════════════

print("=" * 80)
print("PHASE 5: Final evaluation — top 3 models on held-out TEST set")
print("(This is the ONLY time we touch the test set)")
print("=" * 80)

top3_names = res_df["model"].head(3).tolist()
test_results = []

for name in top3_names:
    cfg    = models[name]
    model  = cfg["model"]  # already trained
    use_scaled = cfg["needs_scaling"]
    Xt_final = X_test_sc if use_scaled else X_test

    y_pred = model.predict(Xt_final)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(Xt_final)[:, 1]
    else:
        y_prob = y_pred.astype(float)

    # ─── PATCH 1 (Phase 5): Same corrected routing band ─────────────────
    # Mirrors the fix in Phase 3. Both must stay in sync with ML-SPEC §8.2.
    uncertainty_mask = (y_prob >= 0.30) & (y_prob < 0.85)
    routed_pct = (uncertainty_mask.sum() / len(y_prob)) * 100

    y_pred_calibrated = (y_prob >= PRODUCTION_THRESHOLD).astype(int)
    tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_test, y_pred_calibrated).ravel()
    fpr_calibrated = fp_c / (fp_c + tn_c) if (fp_c + tn_c) > 0 else 0

    acc     = accuracy_score(y_test, y_pred)
    prec    = precision_score(y_test, y_pred)
    rec     = recall_score(y_test, y_pred)
    f1      = f1_score(y_test, y_pred)
    mcc     = matthews_corrcoef(y_test, y_pred)
    auc     = roc_auc_score(y_test, y_prob)
    ap      = average_precision_score(y_test, y_prob)
    ll      = log_loss(y_test, y_prob)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    test_results.append({
        "model": name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "mcc": mcc,
        "auc_roc": auc,
        "avg_precision": ap,
        "log_loss": ll,
        "fpr": fpr,
        "fnr": fnr,
        "calibrated_fpr": fpr_calibrated,
        "routed_pct": routed_pct,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    })

    print(f"\n  ── {name} ──")
    print(f"    Accuracy:          {acc:.5f}")
    print(f"    Balanced Accuracy: {bal_acc:.5f}")
    print(f"    Precision:         {prec:.5f}")
    print(f"    Recall:            {rec:.5f}")
    print(f"    F1 Score:          {f1:.5f}")
    print(f"    MCC:               {mcc:.5f}")
    print(f"    AUC-ROC:           {auc:.5f}")
    print(f"    Avg Precision:     {ap:.5f}")
    print(f"    Log Loss:          {ll:.5f}")
    print(f"    FPR (default):     {fpr:.5f}")
    print(f"    FPR (calibrated):  {fpr_calibrated:.5f}  (threshold={PRODUCTION_THRESHOLD})")
    print(f"    FNR:               {fnr:.5f}")
    print(f"    Enrichment-routed: {routed_pct:.2f}%  (spec §8.2 band [0.30, 0.85))")
    print(f"    Confusion Matrix:  TP={tp:,} TN={tn:,} FP={fp:,} FN={fn:,}")
    print(f"    Classification Report:")
    print(classification_report(y_test, y_pred,
          target_names=["Legitimate", "Phishing"], digits=5))

test_df = pd.DataFrame(test_results).sort_values("mcc", ascending=False)
print("\n  Final Test Leaderboard:")
print(test_df[["model", "accuracy", "f1_score", "mcc", "auc_roc",
               "log_loss", "fpr", "fnr", "calibrated_fpr", "routed_pct"]].to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

# ═══════════════════════════════════════════════════════════════
# PHASE 6 — FEATURE IMPORTANCE FROM BEST MODEL
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 6: Feature importance from the #1 model")
print("=" * 80)

best_name  = test_df.iloc[0]["model"]
best_model = models[best_name]["model"]

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
elif hasattr(best_model, "coef_"):
    importances = np.abs(best_model.coef_[0])
elif hasattr(best_model, "named_estimators_"):
    # Average importances across all base estimators that support it.
    # Using only the first (via break) would misrepresent the ensemble.
    importances = np.zeros(len(feature_names))
    count = 0
    for est_name, est in best_model.named_estimators_.items():
        if hasattr(est, "feature_importances_"):
            importances += est.feature_importances_
            count += 1
    if count > 0:
        importances /= count
        print(f"  (Using averaged importances from {count} tree-based base estimators)")
    else:
        importances = None
else:
    importances = None

if importances is not None:
    feat_imp = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False).reset_index(drop=True)
    feat_imp.index += 1
    feat_imp.index.name = "rank"
    feat_imp["cumulative_pct"] = (feat_imp["importance"].cumsum() /
                                   feat_imp["importance"].sum() * 100)

    print(f"\n  Feature importance from {best_name}:\n")
    print(feat_imp.to_string(float_format=lambda x: f"{x:.6f}"))

    # Which tier are the top features from?
    tier1 = {"url_length", "num_dots", "num_subdomains", "has_ip_address",
             "num_hyphens_url", "num_hyphens_hostname", "https_flag",
             "entropy_url", "num_numeric_chars", "num_sensitive_words"}
    tier2 = {"hostname_length", "path_length", "url_char_prob",
             "char_continuation_rate", "tld_legit_prob", "entropy_domain",
             "num_query_params", "num_special_chars", "at_symbol_present",
             "pct_numeric_chars"}

    top10_feats  = set(feat_imp.head(10)["feature"])
    t1_in_top10  = top10_feats & tier1
    t2_in_top10  = top10_feats & tier2

    print(f"\n  Top-10 features breakdown:")
    print(f"    Tier 1 (paper consensus): {len(t1_in_top10)} — {sorted(t1_in_top10)}")
    print(f"    Tier 2 (strong evidence): {len(t2_in_top10)} — {sorted(t2_in_top10)}")
    print(f"    Tier 3 (useful):          {10 - len(t1_in_top10) - len(t2_in_top10)}")
else:
    print(f"  ⚠ Could not extract feature importances from {best_name}")

# ═══════════════════════════════════════════════════════════════
# PHASE 7 — SAVE ALL RESULTS
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 7: Saving results")
print("=" * 80)

res_df.to_csv(RESULTS_PATH, index=True)
print(f"  ✓ Validation results saved to: {RESULTS_PATH}")

test_path = RESULTS_PATH.replace("benchmark_results", "test_results")
test_df.to_csv(test_path, index=False)
print(f"  ✓ Test results saved to: {test_path}")

# ═══════════════════════════════════════════════════════════════
# PHASE 8 — ANALYSIS & RECOMMENDATIONS
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 8: Analysis & recommendations")
print("=" * 80)

winner = test_df.iloc[0]
print(f"""
  ┌──────────────────────────────────────────────────────────────────┐
  │ CHAMPION MODEL: {winner['model']:47s}│
  │                                                                  │
  │ Test Set Performance:                                            │
  │   Accuracy:         {winner['accuracy']:.5f}                                   │
  │   F1 Score:         {winner['f1_score']:.5f}                                   │
  │   MCC:              {winner['mcc']:.5f}                                   │
  │   AUC-ROC:          {winner['auc_roc']:.5f}                                   │
  │   FPR (default):    {winner['fpr']:.5f}  (legit URLs blocked)           │
  │   FPR (calibrated): {winner['calibrated_fpr']:.5f}  (threshold={PRODUCTION_THRESHOLD})              │
  │   FNR:              {winner['fnr']:.5f}  (phishing URLs missed)         │
  │   Enrichment-routed:{winner['routed_pct']:.2f}%  (spec §8.2 band [0.30, 0.85))  │
  └──────────────────────────────────────────────────────────────────┘
""")

# Comparison with paper results
print("  Paper comparison (on different datasets, for context):")
print("  ┌────────────────────────────────────┬──────────┬───────────┐")
print("  │ Model & Source                      │ Accuracy │ Our Acc.  │")
print("  ├────────────────────────────────────┼──────────┼───────────┤")
print(f"  │ RF — Chiew (2019) HEFS baseline    │  94.60%  │           │")
print(f"  │ RF — PhiUSIIL Table 3              │  99.98%  │           │")

for _, row in test_df.iterrows():
    name    = row["model"]
    our_acc = row["accuracy"] * 100
    print(f"  │ {name:36s} │          │ {our_acc:7.3f}%  │")

print("  └────────────────────────────────────┴──────────┴───────────┘")

# ─── PATCH 4: Corrected next steps ──────────────────────────────────────
print(f"""
  Key observations:
  ─────────────────
  1. The models are trained on OUR PullDD dataset (299K URLs, pruned feature
     set) extracted from raw URLs — not on the same data the papers used.
     Direct accuracy comparison with paper numbers is contextual, not
     apples-to-apples.

  2. Routing band corrected to spec §8.2: enrichment zone is [0.30, 0.85),
     covering both UNCERTAIN (0.30–0.50) and SUSPICIOUS (0.50–0.85) tiers.
     The prior (0.3, 0.7) band left the 0.70–0.85 range unrouted and
     understated routed_pct. Champion routes {winner['routed_pct']:.2f}% of traffic.

  3. Next steps:
     ┌─ PRIORITY ──────────────────────────────────────────────────────────┐
     │ Implement a URL similarity index against a brand lexicon.           │
     │ Error analysis: 58/61 FNs are bare-domain combosquats              │
     │ (e.g. bendigobaniking.com, amznn.net) with median prob=0.003 —     │
     │ structurally indistinguishable from legitimate homepages using      │
     │ URL-only features. These 56 URLs have prob < 0.30 and BYPASS the   │
     │ enrichment cascade entirely. Threshold tuning cannot reach them.   │
     │ Levenshtein / Jaro-Winkler distance to a curated brand list is     │
     │ the correct fix. See PhiUSIIL §4.1 (Algorithm 2, Eq. 2) for the   │
     │ USI approach used by Prasad & Chandra.                             │
     └────────────────────────────────────────────────────────────────────┘

     • Dataset augmentation is already complete — LegitPhish (DS3) is
       fully incorporated into PullDD (63,936 rows). The 45.2/54.8 class
       split is reasonable for training; production threshold calibration
       is a deployment-time concern, not a data gap.

     • Export champion model via joblib for SVC-03 integration.
     • Monitor for concept drift as phishing tactics evolve (PhiUSIIL
       incremental learning approach, §4.2).
""")

PHASE 0: Loading PullDD dataset


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/saifeldenismail/pulldd-phishing-url-low-latency-detection-dataset/cybersiren_lowlatency_dataset(2).csv'

In [5]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║  CyberSiren — Model Selection & Benchmarking Pipeline                ║
║  Tests 10 models on PullDD dataset (299,306 URLs × 30 features)   ║
║  Paper-driven model selection + one bleeding-edge approach         ║
╚══════════════════════════════════════════════════════════════════════╝

Model rationale from literature:
────────────────────────────────
 • Random Forest    — Chiew (2019): "consistently outperform all remaining
                      classifiers by scoring above 95% accuracy."
                      PhiUSIIL Table 3: 99.982% accuracy.
 • XGBoost          — PhiUSIIL Table 3: 99.993% (highest of all tested).
 • LightGBM         — PhiUSIIL Table 3: 99.99% (fastest boosting method).
 • CatBoost         — PhiUSIIL Table 3: 99.987%.
 • Logistic Reg.    — PhiUSIIL Table 3: 99.654% (strong linear baseline).
 • Decision Tree    — Chiew (2019): C4.5 at 94.37%. Simple interpretable.
 • Extra Trees      — Randomized RF variant, often better generalization.
 • AdaBoost         — PhiUSIIL Table 3: 99.981%.
 • Voting Ensemble  — PhiUSIIL §4.2.2: "ensemble of multiple ML algorithms
                      has shown better classification." Their BestSecurity
                      uses 2/3 consensus voting.
 • Stacking         — PhiUSIIL Table 3: 99.979%. "Bleeding edge" approach:
                      stack the top-3 base models with LR meta-learner.
"""

import pandas as pd
import numpy as np
import time
import warnings
import os
import json
from collections import OrderedDict

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve,
    average_precision_score, log_loss, balanced_accuracy_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
    VotingClassifier, StackingClassifier, GradientBoostingClassifier
)

# Boosting libraries
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠ xgboost not available — skipping XGBoost")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("⚠ lightgbm not available — skipping LightGBM")

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("⚠ catboost not available — skipping CatBoost")

warnings.filterwarnings("ignore")
np.random.seed(42)

# ═══════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════

DATASET_PATH = "/kaggle/input/datasets/saifeldenismail/pulldd-phishing-url-low-latency-detection-dataset/cybersiren_lowlatency_dataset(2).csv"
# DATASET_AUGMENTATION_PATH = "/kaggle/input/datasets/legitphish.csv" # TODO: Integrate LegitPhish here to fix class imbalance
RESULTS_PATH = "/kaggle/working/cybersiren_benchmark_results.csv"

# Split ratios — standard 70/15/15
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_STATE = 42

# ─── Production Profiles ───
# Calibrated for real-world traffic where Legitimate to Phishing is ~99:1
PRODUCTION_THRESHOLD = 0.85  # Aggressively suppress FPs on legit blogs/articles

# ═══════════════════════════════════════════════════════════════
# PHASE 0 — LOAD & PREPARE DATA
# ═══════════════════════════════════════════════════════════════

print("=" * 80)
print("PHASE 0: Loading PullDD dataset")
print("=" * 80)

df = pd.read_csv(DATASET_PATH, low_memory=False)
print(f"  Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

# Identify feature columns (everything except url, label, source)
META_COLS = ["url", "label", "source"]
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]
print(f"  Features: {len(FEATURE_COLS)}")
print(f"  Label distribution: {dict(df['label'].value_counts())}")

# ─── Data quality pre-check ───
print("\n  Pre-check:")
zero_var_cols = [c for c in FEATURE_COLS if df[c].std() == 0]
if zero_var_cols:
    print(f"  ⚠ Dropping zero-variance features: {zero_var_cols}")
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in zero_var_cols]
    print(f"  Features after drop: {len(FEATURE_COLS)}")

# ─── IMPROVEMENT: Feature Schema Pruning (HEFS / Obsolete features) ───
# Safely remove features that consume memory space without contributing actionable signal for modern phishing
obsolete_cols = ["has_ip_address", "double_slash_in_path"]
cols_to_prune = [c for c in obsolete_cols if c in FEATURE_COLS]
if cols_to_prune:
    print(f"  ⚠ Pruning obsolete features (HEFS): {cols_to_prune}")
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in cols_to_prune]
    print(f"  Features after pruning: {len(FEATURE_COLS)}")

# Check for any remaining nulls/infs
null_count = df[FEATURE_COLS].isnull().sum().sum()
inf_count = np.isinf(df[FEATURE_COLS].select_dtypes(include=[np.number])).sum().sum()
print(f"  Nulls: {null_count}, Infs: {inf_count}")
assert null_count == 0 and inf_count == 0, "Data quality issue!"
print("  ✓ Data quality passed")

X = df[FEATURE_COLS]  
y = df["label"]       
feature_names = FEATURE_COLS

print(f"\n  X shape: {X.shape}")
print(f"  y shape: {y.shape}, pos_rate: {y.mean():.4f}")

# ═══════════════════════════════════════════════════════════════
# PHASE 1 — TRAIN / VAL / TEST SPLIT
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 1: Stratified train/val/test split (70/15/15)")
print("=" * 80)

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=(VAL_RATIO + TEST_RATIO),
    stratify=y, random_state=RANDOM_STATE
)

# Second split: split temp 50/50 into val and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5,
    stratify=y_temp, random_state=RANDOM_STATE
)

print(f"  Train: {X_train.shape[0]:,} ({X_train.shape[0]/len(X)*100:.1f}%) "
      f"— pos_rate: {y_train.mean():.4f}")
print(f"  Val:   {X_val.shape[0]:,} ({X_val.shape[0]/len(X)*100:.1f}%) "
      f"— pos_rate: {y_val.mean():.4f}")
print(f"  Test:  {X_test.shape[0]:,} ({X_test.shape[0]/len(X)*100:.1f}%) "
      f"— pos_rate: {y_test.mean():.4f}")

scaler = StandardScaler().set_output(transform="pandas")
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

# ═══════════════════════════════════════════════════════════════
# PHASE 2 — MODEL DEFINITIONS
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 2: Defining models")
print("=" * 80)

models = OrderedDict()

models["LogisticRegression"] = {
    "model": LogisticRegression(
        max_iter=1000, C=1.0, solver="lbfgs", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "needs_scaling": True,
    "paper": "PhiUSIIL Table 3: 99.654%",
}

models["DecisionTree"] = {
    "model": DecisionTreeClassifier(
        random_state=RANDOM_STATE, max_depth=None, min_samples_leaf=5
    ),
    "needs_scaling": False,
    "paper": "Chiew (2019): C4.5 at 94.37%",
}

models["RandomForest"] = {
    "model": RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "needs_scaling": False,
    "paper": "Chiew (2019): consensus best. PhiUSIIL: 99.982%",
}

models["ExtraTrees"] = {
    "model": ExtraTreesClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "needs_scaling": False,
    "paper": "RF variant — tested for generalization comparison",
}

models["AdaBoost"] = {
    "model": AdaBoostClassifier(
        n_estimators=200, learning_rate=0.1, random_state=RANDOM_STATE
    ),
    "needs_scaling": False,
    "paper": "PhiUSIIL Table 3: 99.981%",
}

if HAS_XGB:
    _xgb_params = dict(
        n_estimators=300, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        gamma=0.1, reg_alpha=0.01, reg_lambda=1.0,
        eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    try:
        XGBClassifier(use_label_encoder=False, n_estimators=1).get_params()
        _xgb_params["use_label_encoder"] = False
    except TypeError:
        pass  

    models["XGBoost"] = {
        "model": XGBClassifier(**_xgb_params),
        "needs_scaling": False,
        "paper": "PhiUSIIL Table 3: 99.993% — HIGHEST",
    }

if HAS_LGBM:
    models["LightGBM"] = {
        "model": LGBMClassifier(
            n_estimators=300, max_depth=8, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
            reg_alpha=0.01, reg_lambda=1.0, num_leaves=63,
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
        ),
        "needs_scaling": False,
        "paper": "PhiUSIIL Table 3: 99.99%, fastest training",
    }

if HAS_CATBOOST:
    models["CatBoost"] = {
        "model": CatBoostClassifier(
            iterations=300, depth=8, learning_rate=0.1,
            l2_leaf_reg=3.0, random_seed=RANDOM_STATE,
            verbose=0, thread_count=-1
        ),
        "needs_scaling": False,
        "paper": "PhiUSIIL Table 3: 99.987%",
    }

vote_estimators = [("rf", models["RandomForest"]["model"])]
if HAS_XGB:
    vote_estimators.append(("xgb", models["XGBoost"]["model"]))
if HAS_LGBM:
    vote_estimators.append(("lgbm", models["LightGBM"]["model"]))

if len(vote_estimators) >= 2:
    models["VotingEnsemble"] = {
        "model": VotingClassifier(
            estimators=vote_estimators, voting="soft", n_jobs=-1
        ),
        "needs_scaling": False,
        "paper": "PhiUSIIL §4.2.2: multi-model consensus voting",
    }

stack_estimators = [("rf", RandomForestClassifier(
    n_estimators=200, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
))]
if HAS_XGB:
    _xgb_stack = dict(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    try:
        XGBClassifier(use_label_encoder=False, n_estimators=1).get_params()
        _xgb_stack["use_label_encoder"] = False
    except TypeError:
        pass
    stack_estimators.append(("xgb", XGBClassifier(**_xgb_stack)))
if HAS_LGBM:
    stack_estimators.append(("lgbm", LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )))

if len(stack_estimators) >= 2:
    models["StackingEnsemble"] = {
        "model": StackingClassifier(
            estimators=stack_estimators,
            final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            cv=5, stack_method="predict_proba", n_jobs=-1, passthrough=False
        ),
        "needs_scaling": False,
        "paper": "PhiUSIIL Table 3: 99.979%. Bleeding edge: diverse base + LR meta",
    }

print(f"  Models defined: {len(models)}")
for name, cfg in models.items():
    print(f"    • {name:25s} — {cfg['paper']}")

# ═══════════════════════════════════════════════════════════════
# PHASE 3 — TRAINING & EVALUATION
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 3: Training & evaluation on validation set")
print("=" * 80)

results = []

for name, cfg in models.items():
    print(f"\n{'─'*70}")
    print(f"  Training: {name}")
    print(f"  Paper ref: {cfg['paper']}")

    model = cfg["model"]
    use_scaled = cfg["needs_scaling"]

    Xt = X_train_sc if use_scaled else X_train
    Xv = X_val_sc if use_scaled else X_val

    # Train
    t0 = time.time()
    model.fit(Xt, y_train)
    train_time = time.time() - t0

    # Predict on validation
    t0 = time.time()
    y_pred = model.predict(Xv)
    pred_time = time.time() - t0

    # Probabilities for AUC
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(Xv)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = model.decision_function(Xv)
    else:
        y_prob = y_pred.astype(float)

    # ─── IMPROVEMENT: Dynamic Thresholding via Security Profiles ───
    y_pred_calibrated = (y_prob >= PRODUCTION_THRESHOLD).astype(int)
    tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_val, y_pred_calibrated).ravel()
    fpr_calibrated = fp_c / (fp_c + tn_c) if (fp_c + tn_c) > 0 else 0

    # ─── IMPROVEMENT: Cascading Architecture (Uncertainty Routing) ───
    # Routing requests between 0.3 and 0.7 to high-latency service
    uncertainty_mask = (y_prob > 0.3) & (y_prob < 0.7)
    routed_pct = (uncertainty_mask.sum() / len(y_prob)) * 100

    # Metrics
    acc = accuracy_score(y_val, y_pred)
    bal_acc = balanced_accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    mcc = matthews_corrcoef(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    ap = average_precision_score(y_val, y_prob)
    ll = log_loss(y_val, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0 
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0 

    # Batch latency (vectorized — throughput metric)
    batch_latency_us = (pred_time / len(y_val)) * 1e6

    # Single-inference latency (production API reality)
    sample_size = min(1000, len(Xv))
    Xv_sample = Xv.iloc[:sample_size] if hasattr(Xv, "iloc") else Xv[:sample_size]
    t0_single = time.time()
    for i in range(sample_size):
        row = Xv_sample.iloc[[i]] if hasattr(Xv_sample, "iloc") else Xv_sample[[i]]
        _ = model.predict(row)
    single_latency_us = ((time.time() - t0_single) / sample_size) * 1e6

    result = {
        "model": name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "mcc": mcc,
        "auc_roc": auc,
        "avg_precision": ap,
        "log_loss": ll,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "fpr": fpr,
        "fnr": fnr,
        "calibrated_fpr": fpr_calibrated,     # NEW: Evaluated FPR under strict threshold
        "routed_pct": routed_pct,             # NEW: % sent to high-latency enrichment
        "train_time_s": train_time,
        "pred_time_s": pred_time,
        "batch_latency_us": batch_latency_us,
        "single_latency_us": single_latency_us,
    }
    results.append(result)

    print(f"  ✓ Done in {train_time:.1f}s")
    print(f"    Acc={acc:.5f}  F1={f1:.5f}  MCC={mcc:.5f}  AUC={auc:.5f}")
    print(f"    FPR (Default)={fpr:.5f}  |  FPR (Calibrated threshold > {PRODUCTION_THRESHOLD})={fpr_calibrated:.5f}")
    print(f"    Cascading Routing: {routed_pct:.2f}% of URLs fall in uncertainty band (0.3-0.7)")
    print(f"    Batch={batch_latency_us:.1f}μs/URL  Single={single_latency_us:.1f}μs/URL")

# ═══════════════════════════════════════════════════════════════
# PHASE 4 — VALIDATION LEADERBOARD
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 4: Validation leaderboard (ranked by MCC)")
print("=" * 80)

res_df = pd.DataFrame(results)
res_df = res_df.sort_values("mcc", ascending=False).reset_index(drop=True)
res_df.index = res_df.index + 1  
res_df.index.name = "rank"

# Display key metrics
display_cols = ["model", "accuracy", "f1_score", "mcc", "auc_roc", 
                "fpr", "calibrated_fpr", "routed_pct", "single_latency_us"]
print("\n" + res_df[display_cols].to_string(
    float_format=lambda x: f"{x:.5f}" if isinstance(x, float) and abs(x) < 100 else f"{x:.1f}"
))

# Why MCC for ranking:
print("""
  ┌─────────────────────────────────────────────────────────────────┐
  │ Ranking by MCC (Matthews Correlation Coefficient) because:      │
  │ • Unlike accuracy, MCC is balanced even with class imbalance   │
  │ • Unlike F1, MCC accounts for all four confusion matrix cells  │
  │ • Range [-1, +1] where +1 = perfect, 0 = random, -1 = inverse │
  │ • PhiUSIIL Eq. (7) uses MCC as a primary evaluation metric     │
  └─────────────────────────────────────────────────────────────────┘
""")

# ═══════════════════════════════════════════════════════════════
# PHASE 5 — FINAL EVALUATION ON HELD-OUT TEST SET
# ═══════════════════════════════════════════════════════════════

print("=" * 80)
print("PHASE 5: Final evaluation — top 3 models on held-out TEST set")
print("(This is the ONLY time we touch the test set)")
print("=" * 80)

top3_names = res_df["model"].head(3).tolist()
test_results = []

for name in top3_names:
    cfg = models[name]
    model = cfg["model"]  
    use_scaled = cfg["needs_scaling"]
    Xt_final = X_test_sc if use_scaled else X_test

    y_pred = model.predict(Xt_final)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(Xt_final)[:, 1]
    else:
        y_prob = y_pred.astype(float)
        
    # Recalculate Dynamic Thresholding & Uncertainty for Test Set
    y_pred_calibrated = (y_prob >= PRODUCTION_THRESHOLD).astype(int)
    tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_test, y_pred_calibrated).ravel()
    fpr_calibrated = fp_c / (fp_c + tn_c) if (fp_c + tn_c) > 0 else 0

    uncertainty_mask = (y_prob > 0.3) & (y_prob < 0.7)
    routed_pct = (uncertainty_mask.sum() / len(y_prob)) * 100

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ll = log_loss(y_test, y_prob)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    test_results.append({
        "model": name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "mcc": mcc,
        "auc_roc": auc,
        "avg_precision": ap,
        "log_loss": ll,
        "fpr": fpr,
        "fnr": fnr,
        "calibrated_fpr": fpr_calibrated,
        "routed_pct": routed_pct,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    })

    print(f"\n  ── {name} ──")
    print(f"    Accuracy:          {acc:.5f}")
    print(f"    MCC:               {mcc:.5f}")
    print(f"    AUC-ROC:           {auc:.5f}")
    print(f"    FPR (Default):     {fpr:.5f}")
    print(f"    FPR (Calibrated):  {fpr_calibrated:.5f} (Target metric for reducing FP on clean URLs)")
    print(f"    Cascading Route:   {routed_pct:.2f}% (Sent to high-latency system)")
    print(f"    Confusion Matrix:  TP={tp:,} TN={tn:,} FP={fp:,} FN={fn:,}")

test_df = pd.DataFrame(test_results).sort_values("mcc", ascending=False)
print("\n  Final Test Leaderboard:")
print(test_df[["model", "accuracy", "mcc", "auc_roc", 
               "fpr", "calibrated_fpr", "routed_pct"]].to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

# ═══════════════════════════════════════════════════════════════
# PHASE 6 — FEATURE IMPORTANCE FROM BEST MODEL
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 6: Feature importance from the #1 model")
print("=" * 80)

best_name = test_df.iloc[0]["model"]
best_model = models[best_name]["model"]

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
elif hasattr(best_model, "coef_"):
    importances = np.abs(best_model.coef_[0])
elif hasattr(best_model, "named_estimators_"):
    importances = np.zeros(len(feature_names))
    count = 0
    for est_name, est in best_model.named_estimators_.items():
        if hasattr(est, "feature_importances_"):
            importances += est.feature_importances_
            count += 1
    if count > 0:
        importances /= count
        print(f"  (Using averaged importances from {count} tree-based base estimators)")
    else:
        importances = None
else:
    importances = None

if importances is not None:
    feat_imp = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False).reset_index(drop=True)
    feat_imp.index += 1
    feat_imp.index.name = "rank"
    feat_imp["cumulative_pct"] = (feat_imp["importance"].cumsum() /
                                   feat_imp["importance"].sum() * 100)

    print(f"\n  Feature importance from {best_name}:\n")
    print(feat_imp.to_string(float_format=lambda x: f"{x:.6f}"))

    tier1 = {"url_length", "num_dots", "num_subdomains", "has_ip_address",
             "num_hyphens_url", "num_hyphens_hostname", "https_flag",
             "entropy_url", "num_numeric_chars", "num_sensitive_words"}
    tier2 = {"hostname_length", "path_length", "url_char_prob",
             "char_continuation_rate", "tld_legit_prob", "entropy_domain",
             "num_query_params", "num_special_chars", "at_symbol_present",
             "pct_numeric_chars"}

    top10_feats = set(feat_imp.head(10)["feature"])
    t1_in_top10 = top10_feats & tier1
    t2_in_top10 = top10_feats & tier2

    print(f"\n  Top-10 features breakdown:")
    print(f"    Tier 1 (paper consensus): {len(t1_in_top10)} — {sorted(t1_in_top10)}")
    print(f"    Tier 2 (strong evidence): {len(t2_in_top10)} — {sorted(t2_in_top10)}")
    print(f"    Tier 3 (useful):          {10 - len(t1_in_top10) - len(t2_in_top10)}")
else:
    print(f"  ⚠ Could not extract feature importances from {best_name}")

# ═══════════════════════════════════════════════════════════════
# PHASE 7 — SAVE ALL RESULTS
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 7: Saving results")
print("=" * 80)

res_df.to_csv(RESULTS_PATH, index=True)
print(f"  ✓ Validation results saved to: {RESULTS_PATH}")

test_path = RESULTS_PATH.replace("benchmark_results", "test_results")
test_df.to_csv(test_path, index=False)
print(f"  ✓ Test results saved to: {test_path}")

# ═══════════════════════════════════════════════════════════════
# PHASE 8 — ANALYSIS & RECOMMENDATIONS
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 8: Analysis & recommendations")
print("=" * 80)

winner = test_df.iloc[0]
print(f"""
  ┌──────────────────────────────────────────────────────────────────┐
  │ CHAMPION MODEL: {winner['model']:47s}│
  │                                                                  │
  │ Test Set Performance:                                            │
  │   Accuracy:       {winner['accuracy']:.5f}                                        │
  │   F1 Score:       {winner['f1_score']:.5f}                                        │
  │   MCC:            {winner['mcc']:.5f}                                        │
  │   AUC-ROC:        {winner['auc_roc']:.5f}                                        │
  │   FPR (Default):  {winner['fpr']:.5f}  (legit URLs blocked)                │
  │   FPR (Calibrated):{winner['calibrated_fpr']:.5f}  (New threshold)                     │
  │   Cascading Pct:  {winner['routed_pct']:.2f}%     (Sent to secondary enrichment)      │
  └──────────────────────────────────────────────────────────────────┘
""")

# Comparison with paper results
print("  Paper comparison (on different datasets, for context):")
print("  ┌────────────────────────────────────┬──────────┬───────────┐")
print("  │ Model & Source                       │ Accuracy │ Our Acc.  │")
print("  ├────────────────────────────────────┼──────────┼───────────┤")
print(f"  │ RF — Chiew (2019) HEFS baseline    │  94.60%  │           │")
print(f"  │ RF — PhiUSIIL Table 3              │  99.98%  │           │")

for _, row in test_df.iterrows():
    name = row["model"]
    our_acc = row["accuracy"] * 100
    print(f"  │ {name:36s} │          │ {our_acc:7.3f}%  │")

print("  └────────────────────────────────────┴──────────┴───────────┘")

print(f"""
  Key observations:
  ─────────────────
  1. Feature Pruning Executed: Obsolete fields (`has_ip_address`, `double_slash_in_path`)
     were safely pruned early in Phase 0 preventing memory/training bloat.
     
  2. Production Calibrations Applied:
     • Cascading Gate: {winner['routed_pct']:.2f}% of traffic is falling into the uncertainty
       band (0.3-0.7) and will be routed to your high-latency content fetcher.
     • Dynamic Thresholding: Using a stricter confidence threshold (> {PRODUCTION_THRESHOLD}) 
       drops the FP rate to {winner['calibrated_fpr']:.5f}, protecting legitimate domain access.

  3. Next steps:
     • Consider dataset augmentation mapping (e.g. LegitPhish dataset placeholder added)
       to combat penalization on diverse organizational TLDs.
""")

EXPORTING CHAMPION MODEL
  Champion: LightGBM
  Needs scaling: False
  Model saved: /kaggle/working/cybersiren_model/model.joblib (0.8 MB)
  Config saved: /kaggle/working/cybersiren_model/config.json
  Metrics saved: /kaggle/working/cybersiren_model/metrics.json

  Export complete. Files in /kaggle/working/cybersiren_model/:
    config.json                    2.1 KB
    metrics.json                   0.4 KB
    model.joblib                   817.5 KB

  ┌─────────────────────────────────────────────────────────────────┐
  │ Next: Upload the /kaggle/working/cybersiren_model/ folder as   │
  │ a Kaggle dataset, then use CELL 2 below in any notebook.       │
  └─────────────────────────────────────────────────────────────────┘



In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║  CyberSiren — Error Analysis: Every Mistake the Model Made         ║
║  Run AFTER the benchmark cell (Cell 2).                            ║
║  Requires: df, X_test, y_test, best_model, best_name,             ║
║            feature_names, models, scaler                           ║
╚══════════════════════════════════════════════════════════════════════╝
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════
# PHASE 0 — RECONSTRUCT TEST SET WITH URLs
# ═══════════════════════════════════════════════════════════════

print("=" * 80)
print("ERROR ANALYSIS — Every mistake the champion model made")
print(f"Model: {best_name}")
print("=" * 80)

# X_test is a DataFrame with original df indices preserved.
# Use those indices to pull back the raw URLs and source.
cfg = models[best_name]
Xt = X_test if not cfg["needs_scaling"] else scaler.transform(X_test)

y_pred = best_model.predict(Xt)
y_prob = best_model.predict_proba(Xt)[:, 1]

# Build a full analysis frame: url + label + source + features + predictions
test_full = df.loc[X_test.index, ["url", "label", "source"]].copy()
test_full = test_full.join(X_test)
test_full["y_pred"] = y_pred
test_full["y_prob"] = np.round(y_prob, 6)
test_full["correct"] = (test_full["label"] == test_full["y_pred"]).astype(int)

# Classify error type
def error_type(row):
    if row["correct"] == 1:
        return "TP" if row["label"] == 1 else "TN"
    else:
        return "FP" if row["y_pred"] == 1 else "FN"

test_full["error_type"] = test_full.apply(error_type, axis=1)

errors = test_full[test_full["correct"] == 0].copy()
fp = errors[errors["error_type"] == "FP"].copy()
fn = errors[errors["error_type"] == "FN"].copy()

print(f"\n  Test set size:  {len(test_full):,}")
print(f"  Correct:        {test_full['correct'].sum():,}")
print(f"  Total errors:   {len(errors):,}")
print(f"  False Positives: {len(fp):,} (legitimate URLs flagged as phishing)")
print(f"  False Negatives: {len(fn):,} (phishing URLs missed)")


# ═══════════════════════════════════════════════════════════════
# PHASE 1 — LIST EVERY MISCLASSIFIED URL
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("PHASE 1: Every misclassified URL")
print("=" * 80)

print(f"\n── FALSE POSITIVES ({len(fp)}) — Legitimate URLs incorrectly flagged as phishing ──")
print(f"   These are the MOST DAMAGING errors: real sites get blocked.\n")
for i, (_, row) in enumerate(fp.sort_values("y_prob", ascending=False).iterrows()):
    print(f"  FP-{i+1:03d}  prob={row['y_prob']:.4f}  src={row['source']:10s}  {row['url']}")

print(f"\n── FALSE NEGATIVES ({len(fn)}) — Phishing URLs that slipped through ──")
print(f"   These are phishing URLs the model thought were safe.\n")
for i, (_, row) in enumerate(fn.sort_values("y_prob").iterrows()):
    print(f"  FN-{i+1:03d}  prob={row['y_prob']:.4f}  src={row['source']:10s}  {row['url']}")


# ═══════════════════════════════════════════════════════════════
# PHASE 2 — FEATURE PROFILE: ERRORS vs CORRECT
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 2: Feature profiles — what makes errors look different?")
print("=" * 80)

correct_legit = test_full[(test_full["correct"] == 1) & (test_full["label"] == 0)]
correct_phish = test_full[(test_full["correct"] == 1) & (test_full["label"] == 1)]

def feature_comparison(group_a, group_b, label_a, label_b, features):
    """Compare mean feature values between two groups."""
    rows = []
    for feat in features:
        mean_a = group_a[feat].mean()
        mean_b = group_b[feat].mean()
        diff = mean_a - mean_b
        # How far is group_a from the overall mean of group_b in std units
        std_b = group_b[feat].std()
        z = diff / std_b if std_b > 0 else 0
        rows.append({
            "feature": feat,
            f"mean_{label_a}": round(mean_a, 4),
            f"mean_{label_b}": round(mean_b, 4),
            "diff": round(diff, 4),
            "z_score": round(z, 2),
        })
    return pd.DataFrame(rows).sort_values("z_score", key=abs, ascending=False)

print(f"\n── FP Analysis: What makes these {len(fp)} legit URLs look phishy? ──")
print(f"   Compared against correctly classified legitimate URLs.\n")
if len(fp) > 0:
    fp_profile = feature_comparison(fp, correct_legit, "FP", "correct_legit", feature_names)
    print(fp_profile.to_string(index=False))
    print(f"\n   Interpretation: Features with high |z_score| are the ones pushing")
    print(f"   these legitimate URLs toward the phishing side of the decision boundary.")

print(f"\n── FN Analysis: What makes these {len(fn)} phishing URLs look legitimate? ──")
print(f"   Compared against correctly classified phishing URLs.\n")
if len(fn) > 0:
    fn_profile = feature_comparison(fn, correct_phish, "FN", "correct_phish", feature_names)
    print(fn_profile.to_string(index=False))
    print(f"\n   Interpretation: Features with high |z_score| are the ones pulling")
    print(f"   these phishing URLs toward the legitimate side of the decision boundary.")


# ═══════════════════════════════════════════════════════════════
# PHASE 3 — PROBABILITY DISTRIBUTION OF ERRORS
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 3: How confident was the model on its mistakes?")
print("=" * 80)

print(f"\n  FP probability distribution (should be >0.5, higher = more confident mistake):")
if len(fp) > 0:
    print(f"    min:    {fp['y_prob'].min():.4f}")
    print(f"    25%:    {fp['y_prob'].quantile(0.25):.4f}")
    print(f"    median: {fp['y_prob'].median():.4f}")
    print(f"    75%:    {fp['y_prob'].quantile(0.75):.4f}")
    print(f"    max:    {fp['y_prob'].max():.4f}")
    # Bucket them
    close_calls = fp[fp["y_prob"] < 0.6]
    moderate = fp[(fp["y_prob"] >= 0.6) & (fp["y_prob"] < 0.85)]
    confident = fp[fp["y_prob"] >= 0.85]
    print(f"    Close calls (0.50–0.60):   {len(close_calls)}")
    print(f"    Moderate (0.60–0.85):       {len(moderate)}")
    print(f"    Confident mistakes (≥0.85): {len(confident)}")

print(f"\n  FN probability distribution (should be <0.5, lower = more confident mistake):")
if len(fn) > 0:
    print(f"    min:    {fn['y_prob'].min():.4f}")
    print(f"    25%:    {fn['y_prob'].quantile(0.25):.4f}")
    print(f"    median: {fn['y_prob'].median():.4f}")
    print(f"    75%:    {fn['y_prob'].quantile(0.75):.4f}")
    print(f"    max:    {fn['y_prob'].max():.4f}")
    close_calls = fn[fn["y_prob"] > 0.4]
    moderate = fn[(fn["y_prob"] <= 0.4) & (fn["y_prob"] > 0.15)]
    confident = fn[fn["y_prob"] <= 0.15]
    print(f"    Close calls (0.40–0.50):   {len(close_calls)}")
    print(f"    Moderate (0.15–0.40):       {len(moderate)}")
    print(f"    Confident mistakes (≤0.15): {len(confident)}")


# ═══════════════════════════════════════════════════════════════
# PHASE 4 — URL STRUCTURAL PATTERNS IN ERRORS
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 4: Structural patterns in misclassified URLs")
print("=" * 80)

# Extract URL components for pattern analysis
from urllib.parse import urlparse
import re

def extract_url_parts(url_str):
    try:
        parsed = urlparse(url_str if "://" in url_str else f"http://{url_str}")
        hostname = (parsed.hostname or "").lower()
        parts = hostname.rstrip(".").split(".")
        tld = parts[-1] if len(parts) >= 2 else ""
        # Check for double TLDs
        known_double = {"co.uk", "com.au", "co.in", "co.jp", "com.br", "co.za",
                       "com.mx", "org.uk", "net.au", "ac.uk", "gov.uk"}
        if len(parts) >= 3:
            candidate = f"{parts[-2]}.{parts[-1]}"
            if candidate in known_double:
                tld = candidate
        scheme = (parsed.scheme or "").lower()
        path = parsed.path or ""
        return {"tld": tld, "scheme": scheme, "hostname": hostname, "path_ext": path.split(".")[-1] if "." in path else ""}
    except Exception:
        return {"tld": "", "scheme": "", "hostname": "", "path_ext": ""}

print(f"\n── TLD distribution in False Positives ──")
if len(fp) > 0:
    fp_parts = fp["url"].apply(extract_url_parts).apply(pd.Series)
    tld_counts = fp_parts["tld"].value_counts()
    print(f"   {dict(tld_counts.head(15))}")

print(f"\n── TLD distribution in False Negatives ──")
if len(fn) > 0:
    fn_parts = fn["url"].apply(extract_url_parts).apply(pd.Series)
    tld_counts = fn_parts["tld"].value_counts()
    print(f"   {dict(tld_counts.head(15))}")

print(f"\n── Scheme distribution ──")
if len(fp) > 0:
    print(f"   FP: {dict(fp_parts['scheme'].value_counts())}")
if len(fn) > 0:
    print(f"   FN: {dict(fn_parts['scheme'].value_counts())}")

print(f"\n── Source dataset distribution ──")
print(f"   FP: {dict(fp['source'].value_counts())}")
print(f"   FN: {dict(fn['source'].value_counts())}")

# URL length buckets
print(f"\n── URL length distribution ──")
def length_bucket(l):
    if l < 20: return "<20"
    elif l < 30: return "20-29"
    elif l < 50: return "30-49"
    elif l < 75: return "50-74"
    elif l < 100: return "75-99"
    else: return "100+"

if len(fp) > 0:
    fp_len = fp["url_length"].apply(length_bucket).value_counts().sort_index()
    print(f"   FP: {dict(fp_len)}")
if len(fn) > 0:
    fn_len = fn["url_length"].apply(length_bucket).value_counts().sort_index()
    print(f"   FN: {dict(fn_len)}")


# ═══════════════════════════════════════════════════════════════
# PHASE 5 — COMMON PATTERNS AND CLUSTERS
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 5: Common patterns in errors")
print("=" * 80)

# Check for sensitive word presence
print(f"\n── Sensitive words in errors ──")
SENSITIVE_WORDS = ["secure", "account", "webscr", "login", "ebayisapi", "signin",
                   "banking", "confirm", "update", "verify", "password", "suspend",
                   "paypal", "authenticate", "wallet", "credential"]

if len(fp) > 0:
    fp_sw = fp["url"].str.lower().apply(lambda u: [w for w in SENSITIVE_WORDS if w in u])
    fp_has_sw = fp_sw[fp_sw.str.len() > 0]
    print(f"   FP with sensitive words: {len(fp_has_sw)}/{len(fp)}")
    if len(fp_has_sw) > 0:
        all_sw = [w for words in fp_has_sw for w in words]
        from collections import Counter
        print(f"   FP sensitive word freq: {dict(Counter(all_sw).most_common(10))}")

if len(fn) > 0:
    fn_sw = fn["url"].str.lower().apply(lambda u: [w for w in SENSITIVE_WORDS if w in u])
    fn_has_sw = fn_sw[fn_sw.str.len() > 0]
    print(f"   FN with sensitive words: {len(fn_has_sw)}/{len(fn)}")

# Check for IP addresses in errors
print(f"\n── IP-based URLs in errors ──")
print(f"   FP with IP: {fp['has_ip_address'].sum()}")
print(f"   FN with IP: {fn['has_ip_address'].sum()}")

# Check HTTPS
print(f"\n── HTTPS distribution in errors ──")
print(f"   FP with HTTPS: {fp['https_flag'].sum()}/{len(fp)} ({fp['https_flag'].mean()*100:.1f}%)")
print(f"   FN with HTTPS: {fn['https_flag'].sum()}/{len(fn)} ({fn['https_flag'].mean()*100:.1f}%)")
print(f"   Overall legit HTTPS rate:  {correct_legit['https_flag'].mean()*100:.1f}%")
print(f"   Overall phish HTTPS rate:  {correct_phish['https_flag'].mean()*100:.1f}%")

# Check short vs long URLs
print(f"\n── Hostname patterns in False Negatives (phishing that looks legit) ──")
if len(fn) > 0:
    fn_hosts = fn["url"].apply(lambda u: urlparse(u if "://" in u else f"http://{u}").hostname or "")
    # Check for brand names in hostnames
    brands = ["google", "apple", "microsoft", "paypal", "amazon", "facebook", "netflix",
              "bank", "secure", "login", "account", "verify", "update"]
    fn_brand = fn_hosts.apply(lambda h: [b for b in brands if b in h.lower()])
    fn_has_brand = fn_brand[fn_brand.str.len() > 0]
    print(f"   FN with brand-like hostname: {len(fn_has_brand)}/{len(fn)}")
    if len(fn_has_brand) > 0:
        all_brands = [b for bl in fn_has_brand for b in bl]
        print(f"   Brand freq: {dict(Counter(all_brands).most_common(10))}")


# ═══════════════════════════════════════════════════════════════
# PHASE 6 — TOP FEATURES DRIVING EACH INDIVIDUAL ERROR
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 6: Per-error feature deviation analysis")
print("   For each error, which features deviate most from their class mean?")
print("=" * 80)

# Compute class-level statistics
legit_means = correct_legit[feature_names].mean()
legit_stds = correct_legit[feature_names].std().replace(0, 1)  # avoid div by 0
phish_means = correct_phish[feature_names].mean()
phish_stds = correct_phish[feature_names].std().replace(0, 1)

def get_top_deviations(row, reference_means, reference_stds, n=5):
    """For a single row, find the n features most deviating from reference."""
    z_scores = {}
    for feat in feature_names:
        z = (row[feat] - reference_means[feat]) / reference_stds[feat]
        z_scores[feat] = round(z, 2)
    # Sort by absolute z-score
    sorted_z = sorted(z_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    return sorted_z[:n]

print(f"\n── False Positives: Top-5 deviating features per error ──")
print(f"   (These legit URLs deviate from legit norms in these features)\n")
if len(fp) > 0:
    for i, (idx, row) in enumerate(fp.sort_values("y_prob", ascending=False).iterrows()):
        deviations = get_top_deviations(row, legit_means, legit_stds)
        dev_str = "  ".join([f"{f}={z:+.1f}σ" for f, z in deviations])
        print(f"  FP-{i+1:03d} (p={row['y_prob']:.3f}): {dev_str}")
        print(f"          URL: {row['url'][:100]}")

print(f"\n── False Negatives: Top-5 deviating features per error ──")
print(f"   (These phishing URLs deviate from phishing norms in these features)\n")
if len(fn) > 0:
    for i, (idx, row) in enumerate(fn.sort_values("y_prob").iterrows()):
        deviations = get_top_deviations(row, phish_means, phish_stds)
        dev_str = "  ".join([f"{f}={z:+.1f}σ" for f, z in deviations])
        print(f"  FN-{i+1:03d} (p={row['y_prob']:.3f}): {dev_str}")
        print(f"          URL: {row['url'][:100]}")


# ═══════════════════════════════════════════════════════════════
# PHASE 7 — AGGREGATE FEATURE BLAME RANKING
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 7: Feature blame ranking — which features cause the most errors?")
print("=" * 80)

def blame_ranking(error_df, reference_means, reference_stds):
    """Count how often each feature appears in top-3 deviations across all errors."""
    blame_count = {f: 0 for f in feature_names}
    blame_sum = {f: 0.0 for f in feature_names}
    for _, row in error_df.iterrows():
        devs = get_top_deviations(row, reference_means, reference_stds, n=3)
        for feat, z in devs:
            blame_count[feat] += 1
            blame_sum[feat] += abs(z)
    blame_df = pd.DataFrame({
        "feature": list(blame_count.keys()),
        "times_in_top3": list(blame_count.values()),
        "total_abs_z": [round(v, 1) for v in blame_sum.values()],
    })
    blame_df["avg_abs_z"] = (blame_df["total_abs_z"] / blame_df["times_in_top3"].clip(lower=1)).round(2)
    return blame_df.sort_values("times_in_top3", ascending=False)

print(f"\n── FP Blame Ranking (which features make legit URLs look phishy?) ──\n")
if len(fp) > 0:
    fp_blame = blame_ranking(fp, legit_means, legit_stds)
    print(fp_blame[fp_blame["times_in_top3"] > 0].to_string(index=False))

print(f"\n── FN Blame Ranking (which features make phishing URLs look legit?) ──\n")
if len(fn) > 0:
    fn_blame = blame_ranking(fn, phish_means, phish_stds)
    print(fn_blame[fn_blame["times_in_top3"] > 0].to_string(index=False))


# ═══════════════════════════════════════════════════════════════
# PHASE 8 — DECISION BOUNDARY ANALYSIS
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 8: Decision boundary — would a different threshold fix errors?")
print("=" * 80)

from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
print(f"\n  {'Threshold':>10s}  {'Acc':>8s}  {'F1':>8s}  {'MCC':>8s}  {'FP':>5s}  {'FN':>5s}  {'Total Err':>10s}")
print(f"  {'─'*10}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*5}  {'─'*5}  {'─'*10}")

for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    acc = accuracy_score(y_test, y_t)
    f1 = f1_score(y_test, y_t)
    mcc = matthews_corrcoef(y_test, y_t)
    n_fp = ((y_t == 1) & (y_test == 0)).sum()
    n_fn = ((y_t == 0) & (y_test == 1)).sum()
    total = n_fp + n_fn
    marker = " ◄ current" if t == 0.50 else ""
    print(f"  {t:>10.2f}  {acc:>8.5f}  {f1:>8.5f}  {mcc:>8.5f}  {n_fp:>5d}  {n_fn:>5d}  {total:>10d}{marker}")


# ═══════════════════════════════════════════════════════════════
# PHASE 9 — SAVE ERROR DATASET
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("PHASE 9: Saving error dataset for manual review")
print("=" * 80)

error_output = errors[["url", "label", "source", "y_pred", "y_prob", "error_type"] + feature_names]
error_path = "/kaggle/working/cybersiren_error_analysis.csv"
error_output.to_csv(error_path, index=False)
print(f"  ✓ Saved {len(error_output)} misclassified URLs to: {error_path}")
print(f"    Columns: url, label, source, y_pred, y_prob, error_type, + {len(feature_names)} features")
print(f"    Open this CSV to inspect each error manually.")


# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("SUMMARY — Key findings for model improvement")
print("=" * 80)
print(f"""
  Total errors: {len(errors)} out of {len(test_full):,} ({len(errors)/len(test_full)*100:.3f}%)
  False Positives: {len(fp)} (legit blocked)
  False Negatives: {len(fn)} (phishing missed)
  
  Error CSV saved to: {error_path}
  
  Questions to investigate:
  1. Which features appear most in the blame rankings?
  2. Are most errors close calls (prob near 0.5) or confident mistakes?
  3. Do errors cluster around specific TLDs, URL lengths, or source datasets?
  4. Would threshold adjustment reduce total errors?
  5. Are there missing features that could separate the remaining errors?
""")